In [ ]:
!sudo apt-get install -y fonts-nanum
!sudo fc-cache -fv
!rm ~/.cache/matplotlib -rf

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  fonts-nanum
0 upgraded, 1 newly installed, 0 to remove and 41 not upgraded.
Need to get 10.3 MB of archives.
After this operation, 34.1 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 fonts-nanum all 20200506-1 [10.3 MB]
Fetched 10.3 MB in 2s (4,153 kB/s)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 1.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 
Selecting previously unselected package fonts-nanum.
(Reading database ... 125079 files and dire

# 데이터 불러오기

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.rc('font', family='NanumBarunGothic')

!pip install tspoon
import tspoon as tsp

In [ ]:
!pip install xmltodict
import pandas as pd

import requests
import xml.etree.ElementTree as ET
import xml.dom.minidom
import xmltodict

import json
from urllib.request import urlopen

def getECOS(topic,period,beg,end,item1='?',item2='?',item3='?',item4='?', df=None):
    key = ''
    url = 'https://ecos.bok.or.kr/api/StatisticSearch/'+'OIT9OYFOZDZWLHSBQJWU'+'/xml/kr/'+str(1)+'/'+str(50000)+'/'+topic+'/'+period+'/'+beg+'/'+end \
            +'/'+item1+'/'+item2+'/'+item3+'/'+item4
    # print(url)
    ## call OpenAPI URL
    response = requests.get(url)

    ## get API return value upon the http request
    if response.status_code == 200:
        try:
            contents = response.text
            ecosRoot = ET.fromstring(contents)

            # error check
            if ecosRoot[0].text[:4] in ("INFO","ERRO"):
                print(ecosRoot[0].text + " : " + ecosRoot[1].text)

            else:
                #print(contents)
                #dom = xml.dom.minidom.parse(xml_fname)
                dom = xml.dom.minidom.parseString(contents)
                pretty_xml_as_string = dom.toprettyxml(indent=" ")
                # print(pretty_xml_as_string)
                dic = xmltodict.parse(pretty_xml_as_string)

                n = int(dic['StatisticSearch']['list_total_count']['#text'])
                df_ecos = pd.DataFrame(index = [dic['StatisticSearch']['row'][i]['TIME'] for i in range(n)])
                df_ecos[dic['StatisticSearch']['row'][0]['ITEM_NAME1']] = [float(dic['StatisticSearch']['row'][i]['DATA_VALUE']) for i in range(n)]

                if type(df)==type(pd.DataFrame()):
                    df_ecos = df.merge(df_ecos, left_index=True, right_index=True, how='left')

                return df_ecos

        except Exception as e:
            print(str(e))


def getKOSIS(table,period,beg,end,item, orgId='101', obj1='',obj2='',obj3='',obj4='',obj5='',obj6='',obj7='',obj8='', title=None, title_no='0', debug=False):
     # Korean Statistics
    key = 'MzhiMmQwZTYyMWZiNGM1ZGFkZDJmZTI2OTE1OGU2NzQ='
    url = 'https://kosis.kr/openapi/Param/statisticsParameterData.do?method=getList&apiKey='+key \
            +'&orgId='+orgId\
            +'&tblId='+table \
            +'&prdSe='+period \
            +'&startPrdDe='+beg \
            +'&endPrdDe='+end \
            +'&itmId='+item \
            +'&objL1='+obj1 \
            +'&objL2='+obj2 \
            +'&objL3='+obj3 \
            +'&objL4='+obj4 \
            +'&objL5='+obj5 \
            +'&objL6='+obj6 \
            +'&objL7='+obj7 \
            +'&objL8='+obj8 \
            +'&format=json&jsonVD=Y&outputFields=ITM_NM+PRD_DE+DT'
    res = requests.get(url)
    py_json = res.json()

    data = []
    for v in py_json:
        value = []
        value.append(v['PRD_DE'])

        # ✅ DT 값이 '-' 또는 ''일 때 NaN으로 처리
        try:
            value.append(float(v['DT']))
        except (ValueError, KeyError):
            value.append(np.nan)

        data.append(value)

    df = pd.DataFrame(data, columns=['PRD_DE', 'DT'])

    if debug:
        print(df.head())

    return df


    #get json data from url
    with urlopen(url) as url_:
        json_file = url_.read()

    py_json = json.loads(json_file.decode('utf-8'))


    #data transformation
    data = []

    for i, v in enumerate(py_json):
        value = []
        value.append(v['PRD_DE'])
        value.append(float(v['DT']))

        data.append(value)

    if title is not None:
        title = str(title)
    elif title_no=='0':
        title = v['ITM_NM_ENG']
    else:
        title = v['C'+title_no+'_NM_ENG']

    df_kosis = pd.DataFrame({v['PRD_SE']:[x[0] for x in data], title:[x[1] for x in data]}).set_index(v['PRD_SE'])

    if debug:
        return v

    return df_kosis

In [ ]:
import pandas as pd
import numpy as np
import os
import seaborn as sns

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive/', force_remount=True)

Mounted at /content/gdrive/


In [ ]:
# change directory to where you have your data file!!
%cd gdrive/MyDrive/GDP/데이터

/content/gdrive/MyDrive/GDP/데이터


## GDP

In [ ]:
gdp = getECOS('200Y104', 'Q', '2014Q1', '2025Q2','110311').reset_index()
gdp
gdp = gdp.rename(columns={'기계 및 장비 제조업':'gdp'})
gdp

,index,gdp
0,2014Q1,10373.2
1,2014Q2,10670.1
2,2014Q3,10884.5
3,2014Q4,11070.5
4,2015Q1,10882.6
5,2015Q2,10826.7
6,2015Q3,10806.8
7,2015Q4,10658.1
8,2016Q1,9955.5
9,2016Q2,10140.4


##PPI

In [ ]:
ppi = getECOS('404Y014','Q','2014Q1','2025Q2','311AA').reset_index()
ppi

,index,기계및장비
0,2014Q1,97.79
1,2014Q2,98.08
2,2014Q3,98.02
3,2014Q4,98.09
4,2015Q1,98.53
5,2015Q2,98.48
6,2015Q3,98.51
7,2015Q4,98.48
8,2016Q1,98.61
9,2016Q2,98.61


In [ ]:
ppi = ppi.rename(columns={'기계및장비':'ppi'})

##BSI

In [ ]:
업황실적BSI = getECOS('512Y007','M','201401','202506','AA','C2900').reset_index()
매출실적BSI = getECOS('512Y007','M','201401','202506','AB','C2900').reset_index()
채산성실적BSI = getECOS('512Y007','M','201401','202506','AE','C2900').reset_index()
수출실적BSI= getECOS('512Y007','M','201401','202506','AM','C2900').reset_index()
내수판매실적BSI= getECOS('512Y007','M','201401','202506','AL','C2900').reset_index()
생산실적BSI= getECOS('512Y007','M','201401','202506','AC','C2900').reset_index()
신규수주실적BSI= getECOS('512Y007','M','201401','202506','AD','C2900').reset_index()
제품재고실적BSI= getECOS('512Y007','M','201401','202506','AG','C2900').reset_index()
가동률실적BSI= getECOS('512Y007','M','201401','202506','AK','C2900').reset_index()
생산설비실적BSI= getECOS('512Y007','M','201401','202506','AH','C2900').reset_index()
설비투자실적BSI= getECOS('512Y007','M','201401','202506','AI','C2900').reset_index()
자금사정실적BSI = getECOS('512Y007','M','201401','202506','AO','C2900').reset_index()
인력사정실적BSI = getECOS('512Y007','M','201401','202506','AJ','C2900').reset_index()
원자재구입가격BSI = getECOS('512Y007','M','201401','202506','AN','C2900').reset_index()

업황실적BSI = 업황실적BSI.rename(columns={'업황실적BSI 1)':'업황실적BSI'})
매출실적BSI = 매출실적BSI.rename(columns={'매출실적BSI 2)':'매출실적BSI'})
채산성실적BSI = 채산성실적BSI.rename(columns={'채산성실적BSI 6)':'채산성실적BSI'})
수출실적BSI = 수출실적BSI.rename(columns={'수출실적BSI 2)':'수출실적BSI'})
내수판매실적BSI = 내수판매실적BSI.rename(columns={'내수판매실적BSI 2)':'내수판매실적BSI'})
생산실적BSI = 생산실적BSI.rename(columns={'생산실적BSI 2)':'생산실적BSI'})
신규수주실적BSI = 신규수주실적BSI.rename(columns={'신규수주실적BSI 2)':'신규수주실적BSI'})
제품재고실적BSI = 제품재고실적BSI.rename(columns={'제품재고실적BSI 3)':'제품재고실적BSI'})
가동률실적BSI = 가동률실적BSI.rename(columns={'가동률실적BSI 4)':'가동률실적BSI'})
생산설비실적BSI = 생산설비실적BSI.rename(columns={'생산설비실적BSI 3)':'생산설비실적BSI'})
설비투자실적BSI = 설비투자실적BSI.rename(columns={'설비투자실적BSI 5)':'설비투자실적BSI '})
자금사정실적BSI = 자금사정실적BSI.rename(columns={'자금사정실적BSI 6)':'자금사정실적BSI'})
인력사정실적BSI = 인력사정실적BSI.rename(columns={'인력사정실적BSI 3)':'인력사정실적BSI'})
원자재구입가격BSI = 원자재구입가격BSI.rename(columns={'원자재구입가격BSI 4)':'원자재구입가격BSI'})

In [ ]:
from functools import reduce

# 불러온 BSIDataFrame들을 리스트로 모으기
dfs = [업황실적BSI, 매출실적BSI,수출실적BSI, 내수판매실적BSI, 생산실적BSI, 신규수주실적BSI, 제품재고실적BSI, 가동률실적BSI,설비투자실적BSI, 채산성실적BSI, 자금사정실적BSI, 인력사정실적BSI,원자재구입가격BSI]
dfs = [df.drop(columns=[col for col in df.columns if "Unnamed" in col]) for df in dfs]

# '시점' 기준으로 병합
df_merged = reduce(lambda left, right: pd.merge(left, right, on="index", how="outer"), dfs)
df_merged

,index,업황실적BSI,매출실적BSI,수출실적BSI,내수판매실적BSI,생산실적BSI,신규수주실적BSI,제품재고실적BSI,가동률실적BSI,설비투자실적BSI,채산성실적BSI,자금사정실적BSI,인력사정실적BSI,원자재구입가격실적BSI 4)
0,201401,81.0,81.0,85.0,80.0,85.0,84.0,102.0,87.0,97.0,97.0,86.0,94.0,108.0
1,201402,79.0,85.0,84.0,84.0,86.0,86.0,98.0,85.0,95.0,90.0,89.0,98.0,105.0
2,201403,89.0,87.0,86.0,82.0,92.0,90.0,101.0,89.0,98.0,94.0,89.0,94.0,111.0
3,201404,86.0,90.0,89.0,86.0,86.0,85.0,97.0,85.0,95.0,95.0,85.0,93.0,97.0
4,201405,76.0,88.0,80.0,89.0,90.0,92.0,93.0,90.0,99.0,94.0,92.0,94.0,101.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
133,202502,62.0,70.0,71.0,76.0,73.0,63.0,99.0,71.0,86.0,73.0,74.0,97.0,123.0
134,202503,65.0,72.0,75.0,76.0,75.0,70.0,101.0,77.0,89.0,75.0,79.0,100.0,126.0
135,202504,65.0,74.0,83.0,66.0,79.0,74.0,97.0,78.0,88.0,73.0,74.0,99.0,128.0
136,202505,70.0,76.0,81.0,75.0,81.0,78.0,106.0,85.0,91.0,71.0,81.0,96.0,118.0


In [ ]:
df_merged = df_merged.rename(columns={'원자재구입가격실적BSI 4)':'원자재구입가격BSI'})

In [ ]:
df_merged.isnull().sum()

,0
index,0
업황실적BSI,0
매출실적BSI,0
수출실적BSI,0
내수판매실적BSI,0
생산실적BSI,0
신규수주실적BSI,0
제품재고실적BSI,0
가동률실적BSI,0
설비투자실적BSI,0


In [ ]:
import pandas as pd

# index가 숫자라면 문자열로 변환
df_merged['index'] = df_merged['index'].astype(str)

# 연도와 월 분리
df_merged['연도'] = df_merged['index'].str[:4].astype(int)
df_merged['월'] = df_merged['index'].str[4:6].astype(int)

# 월 → 분기 변환 함수
def month_to_quarter(month):
    if month in [1, 2, 3]:
        return 'Q1'
    elif month in [4, 5, 6]:
        return 'Q2'
    elif month in [7, 8, 9]:
        return 'Q3'
    else:
        return 'Q4'

# 분기 계산
df_merged['분기'] = df_merged['월'].apply(month_to_quarter)

# 분기 인덱스 생성 (예: 2015Q1)
df_merged['index'] = df_merged['연도'].astype(str) + df_merged['분기']

# 분기별 평균 집계
df_quarter = df_merged.groupby('index').mean(numeric_only=True).reset_index()

# 결과 확인
print(df_quarter.head())

    index    업황실적BSI    매출실적BSI    수출실적BSI  내수판매실적BSI    생산실적BSI  신규수주실적BSI  \
0  2014Q1  83.000000  84.333333  85.000000  82.000000  87.666667  86.666667   
1  2014Q2  78.666667  87.000000  84.333333  86.000000  88.000000  87.000000   
2  2014Q3  72.000000  78.333333  85.333333  76.666667  86.333333  83.000000   
3  2014Q4  71.666667  89.333333  93.000000  85.666667  90.000000  85.666667   
4  2015Q1  75.666667  97.000000  98.333333  97.000000  98.000000  96.333333   

    제품재고실적BSI    가동률실적BSI  설비투자실적BSI    채산성실적BSI  자금사정실적BSI  인력사정실적BSI  \
0  100.333333   87.000000   96.666667  93.666667  88.000000  95.333333   
1   95.333333   88.666667   97.000000  93.333333  88.000000  95.000000   
2   99.333333   87.000000   93.000000  90.000000  84.666667  97.333333   
3  103.666667   89.000000   93.666667  88.666667  82.000000  94.333333   
4  104.000000  101.000000   95.333333  90.000000  85.000000  96.000000   

   원자재구입가격BSI      연도     월  
0  108.000000  2014.0   2.0  
1   99.666667  2014.

In [ ]:
df_quarter = df_quarter.drop(columns=['연도', '월'], errors='ignore')

In [ ]:
dfs = [ppi, gdp]
df_final = df_quarter.copy()
key_col = 'index'
for df in dfs:
    df_final = df_final.merge(df, on=key_col, how='left')

df_final

,index,업황실적BSI,매출실적BSI,수출실적BSI,내수판매실적BSI,생산실적BSI,신규수주실적BSI,제품재고실적BSI,가동률실적BSI,설비투자실적BSI,채산성실적BSI,자금사정실적BSI,인력사정실적BSI,원자재구입가격BSI,ppi,gdp
0,2014Q1,83.000000,84.333333,85.000000,82.000000,87.666667,86.666667,100.333333,87.000000,96.666667,93.666667,88.000000,95.333333,108.000000,97.79,10373.2
1,2014Q2,78.666667,87.000000,84.333333,86.000000,88.000000,87.000000,95.333333,88.666667,97.000000,93.333333,88.000000,95.000000,99.666667,98.08,10670.1
2,2014Q3,72.000000,78.333333,85.333333,76.666667,86.333333,83.000000,99.333333,87.000000,93.000000,90.000000,84.666667,97.333333,106.000000,98.02,10884.5
3,2014Q4,71.666667,89.333333,93.000000,85.666667,90.000000,85.666667,103.666667,89.000000,93.666667,88.666667,82.000000,94.333333,107.666667,98.09,11070.5
4,2015Q1,75.666667,97.000000,98.333333,97.000000,98.000000,96.333333,104.000000,101.000000,95.333333,90.000000,85.000000,96.000000,99.333333,98.53,10882.6
5,2015Q2,72.000000,90.333333,97.000000,85.333333,93.000000,88.000000,110.666667,93.333333,94.333333,87.333333,80.000000,96.333333,102.333333,98.48,10826.7
6,2015Q3,62.666667,77.333333,85.333333,71.666667,79.666667,72.666667,105.666667,80.000000,90.333333,87.000000,77.000000,97.000000,106.666667,98.51,10806.8
7,2015Q4,61.666667,77.000000,82.333333,72.000000,80.000000,74.000000,105.666667,77.666667,91.666667,85.000000,75.333333,98.333333,100.000000,98.48,10658.1
8,2016Q1,56.333333,70.666667,77.666667,68.333333,76.333333,70.666667,106.333333,77.333333,89.333333,79.000000,74.000000,102.000000,102.333333,98.61,9955.5
9,2016Q2,62.000000,76.333333,80.000000,73.000000,84.666667,78.333333,104.333333,86.666667,89.666667,84.666667,75.666667,95.333333,112.666667,98.61,10140.4


In [ ]:
df_final.to_csv("기계장비1차.csv")

##생산지수

In [ ]:
생산지수 = getKOSIS('DT_1F02001','Q','201401','202502','T20','101','00','C29')

def to_quarter(x):
    year = str(x)[:4]   # 앞 4자리 → 연도
    q = str(x)[-2:]     # 뒤 2자리 → 분기
    return f"{year}Q{q[-1]}"  # 예: 200904 → 2009Q4

생산지수['index']=생산지수['PRD_DE'].apply(to_quarter)
생산지수 = 생산지수.drop(columns=["PRD_DE"])
생산지수 = 생산지수.rename(columns={'DT': '생산지수'})
생산지수

,생산지수,index
0,94.196,2014Q1
1,94.913,2014Q2
2,95.065,2014Q3
3,98.759,2014Q4
4,94.013,2015Q1
5,92.414,2015Q2
6,93.534,2015Q3
7,90.416,2015Q4
8,88.974,2016Q1
9,90.234,2016Q2


https://kosis.kr/openapi/Param/statisticsParameterData.do?method=getList&apiKey=MGNkMzY3MTdhNjZlMTNiZDMxYjU4NDA4NTdjZWU0YmQ=&itmId=T20+&objL1=C29+&objL2=&objL3=&objL4=&objL5=&objL6=&objL7=&objL8=&format=json&jsonVD=Y&prdSe=Q&startPrdDe=201401&endPrdDe=202502&outputFields=OBJ_NM+NM+ITM_NM+PRD_DE+&orgId=101&tblId=DT_1F02016

##내수/수출출하지수

In [ ]:
수출출하지수 = getKOSIS('DT_1F02016','Q','201401','202502','T21','101','C29')

def to_quarter(x):
    year = str(x)[:4]   # 앞 4자리 → 연도
    q = str(x)[-2:]     # 뒤 2자리 → 분기
    return f"{year}Q{q[-1]}"  # 예: 200904 → 2009Q4

수출출하지수['index']=수출출하지수['PRD_DE'].apply(to_quarter)
수출출하지수 = 수출출하지수.drop(columns=["PRD_DE"])
수출출하지수 = 수출출하지수.rename(columns={'DT': '수출출하지수'})
수출출하지수

,수출출하지수,index
0,99.080,2014Q1
1,102.612,2014Q2
2,100.695,2014Q3
3,103.981,2014Q4
4,95.406,2015Q1
5,95.217,2015Q2
6,96.857,2015Q3
7,90.927,2015Q4
8,95.091,2016Q1
9,91.842,2016Q2


In [ ]:
내수출하지수 = getKOSIS('DT_1F02016','Q','201401','202502','T20','101','C29')
def to_quarter(x):
    year = str(x)[:4]   # 앞 4자리 → 연도
    q = str(x)[-2:]     # 뒤 2자리 → 분기
    return f"{year}Q{q[-1]}"  # 예: 200904 → 2009Q4

내수출하지수['index']=내수출하지수['PRD_DE'].apply(to_quarter)
내수출하지수 = 내수출하지수.drop(columns=["PRD_DE"])
내수출하지수 = 내수출하지수.rename(columns={'DT': '내수출하지수'})
내수출하지수

,내수출하지수,index
0,90.668,2014Q1
1,91.032,2014Q2
2,91.561,2014Q3
3,95.727,2014Q4
4,92.881,2015Q1
5,92.169,2015Q2
6,94.707,2015Q3
7,92.263,2015Q4
8,86.196,2016Q1
9,91.643,2016Q2


##생산능력지수&가동률지수

In [ ]:
생산능력지수 = getKOSIS('DT_1F32001','Q','201401','202502','T10','101','C29')
def to_quarter(x):
    year = str(x)[:4]   # 앞 4자리 → 연도
    q = str(x)[-2:]     # 뒤 2자리 → 분기
    return f"{year}Q{q[-1]}"  # 예: 200904 → 2009Q4

생산능력지수['index']=생산능력지수['PRD_DE'].apply(to_quarter)
생산능력지수 = 생산능력지수.drop(columns=["PRD_DE"])
생산능력지수 = 생산능력지수.rename(columns={'DT': '생산능력지수'})
생산능력지수

,생산능력지수,index
0,98.819,2014Q1
1,99.304,2014Q2
2,99.590,2014Q3
3,100.075,2014Q4
4,97.797,2015Q1
5,98.192,2015Q2
6,99.014,2015Q3
7,99.441,2015Q4
8,99.441,2016Q1
9,99.310,2016Q2


In [ ]:
가동률지수 = getKOSIS('DT_1F32001','Q','201401','202502','T30','101','C29')
def to_quarter(x):
    year = str(x)[:4]   # 앞 4자리 → 연도
    q = str(x)[-2:]     # 뒤 2자리 → 분기
    return f"{year}Q{q[-1]}"  # 예: 200904 → 2009Q4

가동률지수['index']=가동률지수['PRD_DE'].apply(to_quarter)
가동률지수 = 가동률지수.drop(columns=["PRD_DE"])
가동률지수 = 가동률지수.rename(columns={'DT': '가동률지수'})
가동률지수

,가동률지수,index
0,110.695,2014Q1
1,114.809,2014Q2
2,112.053,2014Q3
3,111.644,2014Q4
4,107.665,2015Q1
5,106.787,2015Q2
6,104.312,2015Q3
7,101.056,2015Q4
8,100.956,2016Q1
9,102.030,2016Q2


## 수출액/수입액

https://kosis.kr/openapi/Param/statisticsParameterData.do?method=getList&apiKey=MGNkMzY3MTdhNjZlMTNiZDMxYjU4NDA4NTdjZWU0YmQ=&itmId=T001+&objL1=20000+&objL2=&objL3=&objL4=&objL5=&objL6=&objL7=&objL8=&format=json&jsonVD=Y&prdSe=M&startPrdDe=201401&endPrdDe=202506&outputFields=OBJ_NM+NM+ITM_NM+PRD_DE+&orgId=373&tblId=DT_373001_B001

In [ ]:
수출액 = getKOSIS('DT_373001_B001','M','201401','202506','T001','373','20000')
수출액

,PRD_DE,DT
0,201401,3.518386e+09
1,201402,3.491683e+09
2,201403,3.813932e+09
3,201404,4.103635e+09
4,201405,3.834125e+09
...,...,...
133,202502,4.172434e+09
134,202503,4.583673e+09
135,202504,4.759810e+09
136,202505,4.480524e+09


In [ ]:
수입액 = getKOSIS('DT_373001_B001','M','201401','202506','T002','373','20000')
수입액

,PRD_DE,DT
0,201401,2.621683e+09
1,201402,2.929583e+09
2,201403,2.938747e+09
3,201404,3.370108e+09
4,201405,3.161762e+09
...,...,...
133,202502,3.632532e+09
134,202503,3.807077e+09
135,202504,3.924553e+09
136,202505,3.537014e+09


In [ ]:
from functools import reduce

# 불러온 BSIDataFrame들을 리스트로 모으기
df2 = [수출액,수입액]
df2 = [df.drop(columns=[col for col in df.columns if "Unnamed" in col]) for df in df2]

# '시점' 기준으로 병합
df_merged2 = reduce(lambda left, right: pd.merge(left, right, on="PRD_DE", how="outer"), df2)
df_merged2

,PRD_DE,DT_x,DT_y
0,201401,3.518386e+09,2.621683e+09
1,201402,3.491683e+09,2.929583e+09
2,201403,3.813932e+09,2.938747e+09
3,201404,4.103635e+09,3.370108e+09
4,201405,3.834125e+09,3.161762e+09
...,...,...,...
133,202502,4.172434e+09,3.632532e+09
134,202503,4.583673e+09,3.807077e+09
135,202504,4.759810e+09,3.924553e+09
136,202505,4.480524e+09,3.537014e+09


In [ ]:
import pandas as pd

# index가 숫자라면 문자열로 변환
df_merged2['PRD_DE'] = df_merged2['PRD_DE'].astype(str)

# 연도와 월 분리
df_merged2['연도'] = df_merged2['PRD_DE'].str[:4].astype(int)
df_merged2['월'] = df_merged2['PRD_DE'].str[4:6].astype(int)

# 월 → 분기 변환 함수
def month_to_quarter(month):
    if month in [1, 2, 3]:
        return 'Q1'
    elif month in [4, 5, 6]:
        return 'Q2'
    elif month in [7, 8, 9]:
        return 'Q3'
    else:
        return 'Q4'

# 분기 계산
df_merged2['분기'] = df_merged2['월'].apply(month_to_quarter)

# 분기 인덱스 생성 (예: 2015Q1)
df_merged2['index'] = df_merged2['연도'].astype(str) + df_merged2['분기']

# 분기별 평균 집계
df_quarter2 = df_merged2.groupby('index').mean(numeric_only=True).reset_index()

# 결과 확인
print(df_quarter2.head())

    index          DT_x          DT_y      연도     월
0  2014Q1  3.608000e+09  2.830004e+09  2014.0   2.0
1  2014Q2  3.872889e+09  3.116658e+09  2014.0   5.0
2  2014Q3  3.667544e+09  2.858808e+09  2014.0   8.0
3  2014Q4  3.854089e+09  3.166458e+09  2014.0  11.0
4  2015Q1  3.731872e+09  2.921134e+09  2015.0   2.0


In [ ]:
df_quarter2 = df_quarter2.drop(columns=['연도', '월'], errors='ignore')

In [ ]:
df_quarter2 = df_quarter2.rename(columns={'DT_x':'수출액'})
df_quarter2 = df_quarter2.rename(columns={'DT_y':'수입액'})

In [ ]:
df_quarter2.to_csv("기계장비수출입.csv")

## 기업데이터

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df = pd.read_csv('임지오기업.csv')

In [ ]:
기계장비기업 = df[df['매핑한 산업'] == "기계 및 장비 제조업"]

In [ ]:
columns = ['EBITDA', '인건비']
grouped = 기계장비기업.groupby('조사일')[columns].sum()
grouped

,EBITDA,인건비
조사일,,
2000-01-01,0.000000e+00,41070236.0
2000-04-01,0.000000e+00,90555409.0
2000-07-01,0.000000e+00,-7865244.0
2000-10-01,6.921846e+11,837597188.0
2001-01-01,0.000000e+00,59314689.0
...,...,...
2024-04-01,1.207368e+12,461605628.0
2024-07-01,1.155090e+12,441623912.0
2024-10-01,7.714153e+11,782976901.0


In [ ]:
grouped = grouped.reset_index()

In [ ]:
grouped

,조사일,EBITDA,인건비
0,2000-01-01,0.000000e+00,41070236.0
1,2000-04-01,0.000000e+00,90555409.0
2,2000-07-01,0.000000e+00,-7865244.0
3,2000-10-01,6.921846e+11,837597188.0
4,2001-01-01,0.000000e+00,59314689.0
...,...,...,...
97,2024-04-01,1.207368e+12,461605628.0
98,2024-07-01,1.155090e+12,441623912.0
99,2024-10-01,7.714153e+11,782976901.0
100,2025-01-01,1.021859e+12,256544262.0


In [ ]:
grouped['조사일']=pd.to_datetime(grouped['조사일'])

# 분기(Quarter) 단위로 변환
grouped["index"] = grouped["조사일"].dt.to_period("Q")
grouped = grouped.drop(columns=["조사일"])
print(grouped)

           EBITDA          인건비   index
0    0.000000e+00   41070236.0  2000Q1
1    0.000000e+00   90555409.0  2000Q2
2    0.000000e+00   -7865244.0  2000Q3
3    6.921846e+11  837597188.0  2000Q4
4    0.000000e+00   59314689.0  2001Q1
..            ...          ...     ...
97   1.207368e+12  461605628.0  2024Q2
98   1.155090e+12  441623912.0  2024Q3
99   7.714153e+11  782976901.0  2024Q4
100  1.021859e+12  256544262.0  2025Q1
101  1.111777e+12 -249973842.0  2025Q2

[102 rows x 3 columns]


In [ ]:
mask = (grouped["index"] >= pd.Period("2014Q1")) & (grouped["index"] <= pd.Period("2025Q2"))
기업= grouped.loc[mask]
기업
기업['합산'] = 기업['EBITDA']+기업['인건비']
기업

/tmp/ipython-input-1389489221.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  기업['합산'] = 기업['EBITDA']+기업['인건비']


,EBITDA,인건비,index,합산
56,5.437342e+11,2.842432e+08,2014Q1,5.440185e+11
57,5.617213e+11,2.922078e+08,2014Q2,5.620135e+11
58,3.814502e+11,2.689605e+08,2014Q3,3.817191e+11
59,5.230960e+11,1.205299e+09,2014Q4,5.243013e+11
60,5.123197e+11,3.019108e+08,2015Q1,5.126216e+11
61,5.705730e+11,2.773232e+08,2015Q2,5.708503e+11
62,4.574159e+11,2.963735e+08,2015Q3,4.577122e+11
63,3.114078e+11,1.355684e+09,2015Q4,3.127634e+11
64,5.492057e+11,2.468207e+08,2016Q1,5.494525e+11
65,5.412574e+11,2.840172e+08,2016Q2,5.415414e+11


In [ ]:
기업 = 기업[['index'] + [col for col in 기업.columns if col != 'index']]
기업 = 기업.reset_index()

In [ ]:
기업

,level_0,index,EBITDA,인건비,합산
0,56,2014Q1,5.437342e+11,2.842432e+08,5.440185e+11
1,57,2014Q2,5.617213e+11,2.922078e+08,5.620135e+11
2,58,2014Q3,3.814502e+11,2.689605e+08,3.817191e+11
3,59,2014Q4,5.230960e+11,1.205299e+09,5.243013e+11
4,60,2015Q1,5.123197e+11,3.019108e+08,5.126216e+11
5,61,2015Q2,5.705730e+11,2.773232e+08,5.708503e+11
6,62,2015Q3,4.574159e+11,2.963735e+08,4.577122e+11
7,63,2015Q4,3.114078e+11,1.355684e+09,3.127634e+11
8,64,2016Q1,5.492057e+11,2.468207e+08,5.494525e+11
9,65,2016Q2,5.412574e+11,2.840172e+08,5.415414e+11


In [ ]:
기업
기업 = 기업.drop(columns=['level_0'])

In [ ]:
기업

,index,EBITDA,인건비,합산
0,2014Q1,5.437342e+11,2.842432e+08,5.440185e+11
1,2014Q2,5.617213e+11,2.922078e+08,5.620135e+11
2,2014Q3,3.814502e+11,2.689605e+08,3.817191e+11
3,2014Q4,5.230960e+11,1.205299e+09,5.243013e+11
4,2015Q1,5.123197e+11,3.019108e+08,5.126216e+11
5,2015Q2,5.705730e+11,2.773232e+08,5.708503e+11
6,2015Q3,4.574159e+11,2.963735e+08,4.577122e+11
7,2015Q4,3.114078e+11,1.355684e+09,3.127634e+11
8,2016Q1,5.492057e+11,2.468207e+08,5.494525e+11
9,2016Q2,5.412574e+11,2.840172e+08,5.415414e+11


## 전체 데이터 합산

In [ ]:
from functools import reduce
import pandas as pd

# 모든 데이터프레임의 index 컬럼을 문자열로 변환
for df in [df_final,생산지수, 수출출하지수, 내수출하지수,생산능력지수, 가동률지수, df_quarter2,기업]:
    df['index'] = df['index'].astype(str)

# 리스트에 넣고 합치기
dfs = [df_final,생산지수, 수출출하지수, 내수출하지수,생산능력지수, 가동률지수, df_quarter2,기업]
df_merged = reduce(lambda left, right: pd.merge(left, right, on='index', how='outer'), dfs)


In [ ]:
df_merged
df_merged = df_merged[['index'] + [col for col in df_merged.columns if col != 'index']]

In [ ]:
df_merged

,index,업황실적BSI,매출실적BSI,수출실적BSI,내수판매실적BSI,생산실적BSI,신규수주실적BSI,제품재고실적BSI,가동률실적BSI,설비투자실적BSI,...,생산지수,수출출하지수,내수출하지수,생산능력지수,가동률지수,수출액,수입액,EBITDA,인건비,합산
0,2014Q1,83.000000,84.333333,85.000000,82.000000,87.666667,86.666667,100.333333,87.000000,96.666667,...,94.196,99.080,90.668,98.819,110.695,3.608000e+09,2.830004e+09,5.437342e+11,2.842432e+08,5.440185e+11
1,2014Q2,78.666667,87.000000,84.333333,86.000000,88.000000,87.000000,95.333333,88.666667,97.000000,...,94.913,102.612,91.032,99.304,114.809,3.872889e+09,3.116658e+09,5.617213e+11,2.922078e+08,5.620135e+11
2,2014Q3,72.000000,78.333333,85.333333,76.666667,86.333333,83.000000,99.333333,87.000000,93.000000,...,95.065,100.695,91.561,99.590,112.053,3.667544e+09,2.858808e+09,3.814502e+11,2.689605e+08,3.817191e+11
3,2014Q4,71.666667,89.333333,93.000000,85.666667,90.000000,85.666667,103.666667,89.000000,93.666667,...,98.759,103.981,95.727,100.075,111.644,3.854089e+09,3.166458e+09,5.230960e+11,1.205299e+09,5.243013e+11
4,2015Q1,75.666667,97.000000,98.333333,97.000000,98.000000,96.333333,104.000000,101.000000,95.333333,...,94.013,95.406,92.881,97.797,107.665,3.731872e+09,2.921134e+09,5.123197e+11,3.019108e+08,5.126216e+11
5,2015Q2,72.000000,90.333333,97.000000,85.333333,93.000000,88.000000,110.666667,93.333333,94.333333,...,92.414,95.217,92.169,98.192,106.787,3.934809e+09,2.941282e+09,5.705730e+11,2.773232e+08,5.708503e+11
6,2015Q3,62.666667,77.333333,85.333333,71.666667,79.666667,72.666667,105.666667,80.000000,90.333333,...,93.534,96.857,94.707,99.014,104.312,3.544786e+09,2.685010e+09,4.574159e+11,2.963735e+08,4.577122e+11
7,2015Q4,61.666667,77.000000,82.333333,72.000000,80.000000,74.000000,105.666667,77.666667,91.666667,...,90.416,90.927,92.263,99.441,101.056,3.618473e+09,2.674149e+09,3.114078e+11,1.355684e+09,3.127634e+11
8,2016Q1,56.333333,70.666667,77.666667,68.333333,76.333333,70.666667,106.333333,77.333333,89.333333,...,88.974,95.091,86.196,99.441,100.956,3.380938e+09,2.459957e+09,5.492057e+11,2.468207e+08,5.494525e+11
9,2016Q2,62.000000,76.333333,80.000000,73.000000,84.666667,78.333333,104.333333,86.666667,89.666667,...,90.234,91.842,91.643,99.310,102.030,3.526572e+09,2.625121e+09,5.412574e+11,2.840172e+08,5.415414e+11


In [ ]:
df_merged.to_csv('기계장비.csv')

# 데이터QoQ반환

In [ ]:
import pandas as pd

# df는 위의 데이터프레임
df_qoq = df_merged.copy()

# 분기 순서대로 정렬 (혹시 index가 문자열일 경우를 대비)
df_qoq = df_qoq.sort_values('index')

# 수치형 컬럼만 선택 (index 제외)
num_cols = df_qoq.select_dtypes(include=['number']).columns

# 전분기 대비 증감률 (%) 계산
df_qoq[num_cols] = df_qoq[num_cols].pct_change() * 100

# 결과 확인
print(df_qoq.head())


    index   업황실적BSI    매출실적BSI   수출실적BSI  내수판매실적BSI   생산실적BSI  신규수주실적BSI  \
0  2014Q1       NaN        NaN       NaN        NaN       NaN        NaN   
1  2014Q2 -5.220884   3.162055 -0.784314   4.878049  0.380228   0.384615   
2  2014Q3 -8.474576  -9.961686  1.185771 -10.852713 -1.893939  -4.597701   
3  2014Q4 -0.462963  14.042553  8.984375  11.739130  4.247104   3.212851   
4  2015Q1  5.581395   8.582090  5.734767  13.229572  8.888889  12.451362   

   제품재고실적BSI   가동률실적BSI  설비투자실적BSI   ...      생산지수    수출출하지수    내수출하지수  \
0        NaN        NaN         NaN  ...       NaN       NaN       NaN   
1  -4.983389   1.915709    0.344828  ...  0.761179  3.564796  0.401465   
2   4.195804  -1.879699   -4.123711  ...  0.160147 -1.868203  0.581114   
3   4.362416   2.298851    0.716846  ...  3.885762  3.263320  4.549972   
4   0.321543  13.483146    1.779359  ... -4.805638 -8.246699 -2.973038   

     생산능력지수     가동률지수       수출액        수입액     EBITDA         인건비         합산  
0       NaN       N

In [ ]:
df_qoq
df_qoq = df_qoq.drop(index=[0, 1, 2,3])

In [ ]:
df_qoq.isnull().sum()

,0
index,0
업황실적BSI,0
매출실적BSI,0
수출실적BSI,0
내수판매실적BSI,0
생산실적BSI,0
신규수주실적BSI,0
제품재고실적BSI,0
가동률실적BSI,0
설비투자실적BSI,0


In [ ]:
df_qoq.to_csv('기계장비qoq.csv')

#데이터YoY변환

In [ ]:
import pandas as pd

# df_merged는 원본 데이터프레임
df_yoy = df_merged.copy()

# 분기 순서대로 정렬
df_yoy = df_yoy.sort_values('index')

# 수치형 컬럼만 선택 (index 제외)
num_cols = df_yoy.select_dtypes(include=['number']).columns

# 전년 동분기 대비 증감률 (%) 계산 (4분기 차이)
df_yoy[num_cols] = df_yoy[num_cols].pct_change(periods=4) * 100

# 결과 확인
print(df_yoy.head(10))

    index    업황실적BSI    매출실적BSI    수출실적BSI  내수판매실적BSI    생산실적BSI  신규수주실적BSI  \
0  2014Q1        NaN        NaN        NaN        NaN        NaN        NaN   
1  2014Q2        NaN        NaN        NaN        NaN        NaN        NaN   
2  2014Q3        NaN        NaN        NaN        NaN        NaN        NaN   
3  2014Q4        NaN        NaN        NaN        NaN        NaN        NaN   
4  2015Q1  -8.835341  15.019763  15.686275  18.292683  11.787072  11.153846   
5  2015Q2  -8.474576   3.831418  15.019763  -0.775194   5.681818   1.149425   
6  2015Q3 -12.962963  -1.276596   0.000000  -6.521739  -7.722008 -12.449799   
7  2015Q4 -13.953488 -13.805970 -11.469534 -15.953307 -11.111111 -13.618677   
8  2016Q1 -25.550661 -27.147766 -21.016949 -29.553265 -22.108844 -26.643599   
9  2016Q2 -13.888889 -15.498155 -17.525773 -14.453125  -8.960573 -10.984848   

   제품재고실적BSI   가동률실적BSI  설비투자실적BSI   ...      생산지수     수출출하지수    내수출하지수  \
0        NaN        NaN         NaN  ...       NaN     

In [ ]:
df_yoy=df_yoy.dropna()

In [ ]:
df_yoy

,index,업황실적BSI,매출실적BSI,수출실적BSI,내수판매실적BSI,생산실적BSI,신규수주실적BSI,제품재고실적BSI,가동률실적BSI,설비투자실적BSI,...,생산지수,수출출하지수,내수출하지수,생산능력지수,가동률지수,수출액,수입액,EBITDA,인건비,합산
4,2015Q1,-8.835341,15.019763,15.686275,18.292683,11.787072,11.153846,3.654485,16.091954,-1.379310,...,-0.194276,-3.708115,2.440773,-1.034214,-2.737251,3.433255,3.220133,-5.777546,6.215699,-5.771280
5,2015Q2,-8.474576,3.831418,15.019763,-0.775194,5.681818,1.149425,16.083916,5.263158,-2.749141,...,-2.632938,-7.206759,1.249011,-1.119794,-6.987257,1.598820,-5.627045,1.575810,-5.093831,1.572343
6,2015Q3,-12.962963,-1.276596,0.000000,-6.521739,-7.722008,-12.449799,6.375839,-8.045977,-2.867384,...,-1.610477,-3.811510,3.435961,-0.578371,-6.908338,-3.347138,-6.079409,19.914974,10.192200,19.908123
7,2015Q4,-13.953488,-13.805970,-11.469534,-15.953307,-11.111111,-13.618677,1.929260,-12.734082,-2.135231,...,-8.447838,-12.554217,-3.618624,-0.633525,-9.483716,-6.113400,-15.547627,-40.468332,12.476974,-40.346618
8,2016Q1,-25.550661,-27.147766,-21.016949,-29.553265,-22.108844,-26.643599,2.243590,-23.432343,-6.293706,...,-5.359897,-0.330168,-7.197382,1.681033,-6.231366,-9.403692,-15.787610,7.199788,-18.247159,7.184801
9,2016Q2,-13.888889,-15.498155,-17.525773,-14.453125,-8.960573,-10.984848,-5.722892,-7.142857,-4.946996,...,-2.358950,-3.544535,-0.570691,1.138586,-4.454662,-10.375029,-10.749090,-5.137922,2.413780,-5.134253
10,2016Q3,3.723404,-3.017241,-16.796875,2.325581,1.255230,0.000000,-0.630915,0.416667,1.107011,...,0.233070,-4.266083,3.693497,0.066657,-3.220147,-6.164043,5.036533,10.385881,-16.240744,10.368640
11,2016Q4,4.324324,2.597403,-1.214575,7.870370,3.333333,7.207207,-0.630915,6.866953,1.090909,...,12.047646,9.157896,13.618677,-0.198107,0.243429,3.995439,9.600032,158.118481,7.527316,157.465738
12,2017Q1,39.644970,26.415094,12.446352,27.317073,20.087336,24.528302,-1.880878,21.120690,6.343284,...,17.580417,10.679244,22.155320,-1.024728,3.811561,13.846652,36.767962,59.249840,17.936700,59.231282
13,2017Q2,37.096774,21.397380,19.583333,18.264840,11.811024,13.617021,-2.875399,11.923077,5.576208,...,21.695813,20.913090,21.615399,-0.695801,5.009311,24.536583,44.988950,145.473398,4.207509,145.399310


In [ ]:
df_yoy.to_csv("기계장비yoy.csv")

#상관계수 보기(qoq)

In [ ]:
qoq = pd.read_csv('기계장비qoq.csv')
qoq = qoq.drop(columns=["Unnamed: 0"])

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# GDP_QoQ 기준
target = 'gdp'

In [ ]:
# 수치형 컬럼만 선택
num_cols = qoq.select_dtypes(include=['number']).columns

# GDP 기준 상관계수 계산
corr = qoq[num_cols].corr()[target].sort_values(ascending=False)

print("📊 GDP 대비 단순 상관계수")
print(corr)


📊 GDP 대비 단순 상관계수
gdp           1.000000
생산지수          0.718497
내수출하지수        0.708106
매출실적BSI       0.656270
업황실적BSI       0.646005
수출실적BSI       0.614636
생산실적BSI       0.606203
신규수주실적BSI     0.586834
가동률실적BSI      0.576615
내수판매실적BSI     0.565992
수출액           0.475486
수출출하지수        0.472829
자금사정실적BSI     0.467228
설비투자실적BSI     0.416056
가동률지수         0.409211
채산성실적BSI      0.308999
수입액           0.255117
원자재구입가격BSI    0.242673
합산            0.213504
EBITDA        0.212040
인건비           0.114037
생산능력지수        0.040563
ppi           0.008140
제품재고실적BSI    -0.208703
인력사정실적BSI    -0.526901
Name: gdp, dtype: float64


In [ ]:
# df2 사본
df2 = qoq.copy()

# target 및 feature 지정
target_col = 'gdp'
feature_cols = [col for col in df2.columns if col not in ['index', target_col]]

# 결과 저장용 데이터프레임
corr_df = pd.DataFrame(columns=['Feature', 'Lag', 'Correlation'])

# lag 1~4 생성 및 상관계수 계산
for col in feature_cols:
    for lag in range(1, 5):  # lag 1~4
        lag_col_name = f"{col}_lag{lag}"
        df2[lag_col_name] = df2[col].shift(lag)  # 실제 데이터에 lag 컬럼 추가
        corr = df2[target_col].corr(df2[lag_col_name])  # 상관계수 계산
        corr_df = pd.concat(
            [corr_df, pd.DataFrame({'Feature': [col], 'Lag': [lag], 'Correlation': [corr]})],
            ignore_index=True
        )

# 상관계수 높은 순으로 정렬
corr_df.sort_values(by='Correlation', ascending=False, inplace=True)
corr_df.reset_index(drop=True, inplace=True)

corr_df

/tmp/ipython-input-1052179002.py:17: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  corr_df = pd.concat(


,Feature,Lag,Correlation
0,생산능력지수,4,0.355392
1,수출출하지수,1,0.345073
2,원자재구입가격BSI,1,0.327196
3,내수판매실적BSI,1,0.325498
4,생산지수,1,0.322110
...,...,...,...
91,채산성실적BSI,2,-0.195530
92,내수판매실적BSI,4,-0.210515
93,수출실적BSI,4,-0.218137
94,제품재고실적BSI,2,-0.234606


In [ ]:
corr_df.to_csv('k.csv')

In [ ]:
df2

,index,업황실적BSI,매출실적BSI,수출실적BSI,내수판매실적BSI,생산실적BSI,신규수주실적BSI,제품재고실적BSI,가동률실적BSI,설비투자실적BSI,...,EBITDA_lag3,EBITDA_lag4,인건비_lag1,인건비_lag2,인건비_lag3,인건비_lag4,합산_lag1,합산_lag2,합산_lag3,합산_lag4
0,2015Q1,5.581395,8.582090,5.734767,13.229572,8.888889,12.451362,0.321543,13.483146,1.779359,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2015Q2,-4.845815,-6.872852,-1.355932,-12.027491,-5.102041,-8.650519,6.410256,-7.590759,-1.048951,...,NaN,NaN,-74.951371,NaN,NaN,NaN,-2.227654,NaN,NaN,NaN
2,2015Q3,-12.962963,-14.391144,-12.027491,-16.015625,-14.336918,-17.424242,-4.518072,-14.285714,-4.240283,...,NaN,NaN,-8.143997,-74.951371,NaN,NaN,11.358989,-2.227654,NaN,NaN
3,2015Q4,-1.595745,-0.431034,-3.515625,0.465116,0.418410,1.834862,0.000000,-2.916667,1.476015,...,-2.060086,NaN,6.869332,-8.143997,-74.951371,NaN,-19.819214,11.358989,-2.227654,NaN
4,2016Q1,-8.648649,-8.225108,-5.668016,-5.092593,-4.583333,-4.504505,0.630915,-0.429185,-2.545455,...,11.370482,-2.060086,357.424088,6.869332,-8.143997,-74.951371,-31.668108,-19.819214,11.358989,-2.227654
5,2016Q2,10.059172,8.018868,3.004292,6.829268,10.917031,10.849057,-1.880878,12.068966,0.373134,...,-19.832186,11.370482,-81.793637,357.424088,6.869332,-8.143997,75.676703,-31.668108,-19.819214,11.358989
6,2016Q3,4.838710,-1.746725,-11.250000,0.456621,-4.724409,-7.234043,0.638978,-7.307692,1.858736,...,-31.920212,-19.832186,15.070257,-81.793637,357.424088,6.869332,-1.439817,75.676703,-31.668108,-19.819214
7,2016Q4,-1.025641,5.333333,14.553991,5.909091,2.479339,9.174312,0.000000,3.319502,1.459854,...,76.362235,-31.920212,-12.596764,15.070257,-81.793637,357.424088,-6.716128,-1.439817,75.676703,-31.668108
8,2017Q1,22.279793,13.080169,7.377049,12.017167,10.887097,10.924370,-0.634921,12.851406,2.517986,...,-1.447237,76.362235,487.225659,-12.596764,15.070257,-81.793637,59.403261,-6.716128,-1.439817,75.676703
9,2017Q2,8.050847,3.731343,9.541985,-0.766284,3.272727,1.136364,-2.875399,3.558719,-0.350877,...,-6.713042,-1.447237,-80.031136,487.225659,-12.596764,15.070257,8.648346,59.403261,-6.716128,-1.439817


In [ ]:
# 숫자형 컬럼만 선택 (연도분기 제외)
numeric_df = df2.select_dtypes(include=["number"])

# 전체 상관계수
corr = numeric_df.corr()

target_corr = corr["gdp"].drop("gdp")
target_corr = target_corr.sort_values(ascending=False)
target_corr

,gdp
생산지수,0.718497
내수출하지수,0.708106
매출실적BSI,0.656270
업황실적BSI,0.646005
수출실적BSI,0.614636
...,...
내수판매실적BSI_lag4,-0.210515
수출실적BSI_lag4,-0.218137
제품재고실적BSI_lag2,-0.234606
ppi_lag4,-0.332035


In [ ]:
target_corr.to_csv('기계장비순서.csv')

In [ ]:
df2.dropna(inplace=True)

In [ ]:
df2

,index,업황실적BSI,매출실적BSI,수출실적BSI,내수판매실적BSI,생산실적BSI,신규수주실적BSI,제품재고실적BSI,가동률실적BSI,설비투자실적BSI,...,EBITDA_lag3,EBITDA_lag4,인건비_lag1,인건비_lag2,인건비_lag3,인건비_lag4,합산_lag1,합산_lag2,합산_lag3,합산_lag4
4,2016Q1,-8.648649,-8.225108,-5.668016,-5.092593,-4.583333,-4.504505,0.630915,-0.429185,-2.545455,...,11.370482,-2.060086,357.424088,6.869332,-8.143997,-74.951371,-31.668108,-19.819214,11.358989,-2.227654
5,2016Q2,10.059172,8.018868,3.004292,6.829268,10.917031,10.849057,-1.880878,12.068966,0.373134,...,-19.832186,11.370482,-81.793637,357.424088,6.869332,-8.143997,75.676703,-31.668108,-19.819214,11.358989
6,2016Q3,4.838710,-1.746725,-11.250000,0.456621,-4.724409,-7.234043,0.638978,-7.307692,1.858736,...,-31.920212,-19.832186,15.070257,-81.793637,357.424088,6.869332,-1.439817,75.676703,-31.668108,-19.819214
7,2016Q4,-1.025641,5.333333,14.553991,5.909091,2.479339,9.174312,0.000000,3.319502,1.459854,...,76.362235,-31.920212,-12.596764,15.070257,-81.793637,357.424088,-6.716128,-1.439817,75.676703,-31.668108
8,2017Q1,22.279793,13.080169,7.377049,12.017167,10.887097,10.924370,-0.634921,12.851406,2.517986,...,-1.447237,76.362235,487.225659,-12.596764,15.070257,-81.793637,59.403261,-6.716128,-1.439817,75.676703
9,2017Q2,8.050847,3.731343,9.541985,-0.766284,3.272727,1.136364,-2.875399,3.558719,-0.350877,...,-6.713042,-1.447237,-80.031136,487.225659,-12.596764,15.070257,8.648346,59.403261,-6.716128,-1.439817
10,2017Q3,-7.058824,-6.115108,-9.407666,-3.088803,-7.042254,-6.367041,0.986842,-7.903780,-1.408451,...,59.192927,-6.713042,1.674753,-80.031136,487.225659,-12.596764,51.896039,8.648346,59.403261,-6.716128
11,2017Q4,-5.485232,-3.065134,2.307692,-7.968127,-1.515152,-0.400000,0.000000,-4.104478,1.428571,...,8.809170,59.192927,10.827052,1.674753,-80.031136,487.225659,-27.523468,51.896039,8.648346,59.403261
12,2018Q1,1.339286,0.395257,-0.751880,1.731602,0.384615,-0.803213,0.977199,0.000000,-1.408451,...,51.912754,8.809170,491.267167,10.827052,1.674753,-80.031136,-1.142770,-27.523468,51.896039,8.648346
13,2018Q2,-3.524229,-2.755906,-9.090909,4.680851,-0.383142,3.238866,-2.258065,3.112840,-2.142857,...,-27.532011,51.912754,-83.772131,491.267167,10.827052,1.674753,-7.717767,-1.142770,-27.523468,51.896039


In [ ]:
df2 = df2.reset_index()
df2 = df2.drop(columns=["level_0"])

In [ ]:
df2.to_csv('기계장비lag.csv', encoding='utf-8-sig')

#ARIMA

In [ ]:
import pandas as pd
import numpy as np
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_absolute_error
import warnings
warnings.filterwarnings("ignore")

df = pd.read_csv('기계장비lag.csv')

# 분기형 인덱스로 설정
df['index'] = pd.PeriodIndex(df['index'], freq='Q')
df = df.set_index('index')

# -----------------------------
# 2️⃣ 예측 대상(y)과 외생 변수(X) 설정
# -----------------------------
target = 'gdp'   # GDP 예측 대상 칼럼
exog_vars = ['생산지수','내수출하지수','매출실적BSI','업황실적BSI','수출실적BSI','생산실적BSI','신규수주실적BSI',
             '가동률실적BSI','내수판매실적BSI','ppi_lag4','인력사정실적BSI'
]

y = df[target].dropna()
X = df[exog_vars].dropna()

# y와 X의 인덱스 동기화
df_model = pd.concat([y, X], axis=1).dropna()
y = df_model[target]
X = df_model[exog_vars]

# -----------------------------
# 3️⃣ 롤링 예측 설정
# -----------------------------
periods_per_year = 4
train_years = 5
train_window = train_years * periods_per_year

# 예측 구간: 2017Q1 이후부터 2025Q2까지
start_period = pd.Period('2016Q1', freq='Q')
end_period = pd.Period('2025Q2', freq='Q')

# -----------------------------
# 4️⃣ 파라미터 범위 설정
# -----------------------------
p_values = range(0, 4)
d_values = range(1, 4)
q_values = range(0, 4)

# -----------------------------
# 5️⃣ 결과 저장
# -----------------------------
results = []

# -----------------------------
# 6️⃣ (p,d,q) 조합별 탐색
# -----------------------------
for p in p_values:
    for d in d_values:
        for q in q_values:

            train_mae_list = []
            test_mae_list = []

            for test_start in pd.period_range(start_period, end_period, freq='Q'):
                train_end = test_start - 1
                test_end = test_start
                train_start = train_end - (train_window - 1)

                train_data = y.loc[train_start:train_end]
                test_data = y.loc[test_end:test_end]
                X_train = X.loc[train_start:train_end]
                X_test = X.loc[test_end:test_end]

                if len(train_data) < train_window or len(X_test) == 0:
                    continue

                try:
                    model = SARIMAX(train_data, exog=X_train, order=(p, d, q),
                                    enforce_stationarity=False, enforce_invertibility=False)
                    fit = model.fit(disp=False)

                    # 예측
                    pred = fit.forecast(steps=1, exog=X_test)

                    # Train MAE (fittedvalues vs train_data)
                    train_mae = mean_absolute_error(train_data, fit.fittedvalues)
                    train_mae_list.append(train_mae)

                    # Test MAE (out-of-sample)
                    test_mae = mean_absolute_error(test_data, pred)
                    test_mae_list.append(test_mae)

                except Exception:
                    continue

            if train_mae_list and test_mae_list:
                results.append({
                    'p': p, 'd': d, 'q': q,
                    'train_MAE': np.mean(train_mae_list),
                    'test_MAE': np.mean(test_mae_list)
                })

# -----------------------------
# 7️⃣ 결과 정리 및 최적 모델 출력
# -----------------------------
results_df = pd.DataFrame(results)
best = results_df.loc[results_df['test_MAE'].idxmin()]

print("✅ 최적 ARIMAX(p,d,q):", tuple(best[['p', 'd', 'q']]))
print("📉 평균 Train MAE:", round(best['train_MAE'], 4))
print("📈 평균 Test MAE:", round(best['test_MAE'], 4))

✅ 최적 ARIMAX(p,d,q): (1.0, 1.0, 3.0)
📉 평균 Train MAE: 1.5463
📈 평균 Test MAE: 4.5765


In [ ]:
import pandas as pd
import numpy as np
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_absolute_error
import warnings
warnings.filterwarnings("ignore")

df = pd.read_csv('기계장비lag.csv')

# 분기형 인덱스로 설정
df['index'] = pd.PeriodIndex(df['index'], freq='Q')
df = df.set_index('index')

# -----------------------------
# 2️⃣ 예측 대상(y)과 외생 변수(X) 설정
# -----------------------------
target = 'gdp'   # GDP 예측 대상 칼럼
exog_vars = ['생산지수','내수출하지수','매출실적BSI','업황실적BSI','수출실적BSI','생산실적BSI','신규수주실적BSI',
             '가동률실적BSI','내수판매실적BSI','인력사정실적BSI'
]

y = df[target].dropna()
X = df[exog_vars].dropna()

# y와 X의 인덱스 동기화
df_model = pd.concat([y, X], axis=1).dropna()
y = df_model[target]
X = df_model[exog_vars]

# -----------------------------
# 3️⃣ 롤링 예측 설정
# -----------------------------
periods_per_year = 4
train_years = 5
train_window = train_years * periods_per_year

# 예측 구간: 2017Q1 이후부터 2025Q2까지
start_period = pd.Period('2016Q1', freq='Q')
end_period = pd.Period('2025Q2', freq='Q')

# -----------------------------
# 4️⃣ 파라미터 범위 설정
# -----------------------------
p_values = range(0, 4)
d_values = range(1, 4)
q_values = range(0, 4)

# -----------------------------
# 5️⃣ 결과 저장
# -----------------------------
results = []

# -----------------------------
# 6️⃣ (p,d,q) 조합별 탐색
# -----------------------------
for p in p_values:
    for d in d_values:
        for q in q_values:

            train_mae_list = []
            test_mae_list = []

            for test_start in pd.period_range(start_period, end_period, freq='Q'):
                train_end = test_start - 1
                test_end = test_start
                train_start = train_end - (train_window - 1)

                train_data = y.loc[train_start:train_end]
                test_data = y.loc[test_end:test_end]
                X_train = X.loc[train_start:train_end]
                X_test = X.loc[test_end:test_end]

                if len(train_data) < train_window or len(X_test) == 0:
                    continue

                try:
                    model = SARIMAX(train_data, exog=X_train, order=(p, d, q),
                                    enforce_stationarity=False, enforce_invertibility=False)
                    fit = model.fit(disp=False)

                    # 예측
                    pred = fit.forecast(steps=1, exog=X_test)

                    # Train MAE (fittedvalues vs train_data)
                    train_mae = mean_absolute_error(train_data, fit.fittedvalues)
                    train_mae_list.append(train_mae)

                    # Test MAE (out-of-sample)
                    test_mae = mean_absolute_error(test_data, pred)
                    test_mae_list.append(test_mae)

                except Exception:
                    continue

            if train_mae_list and test_mae_list:
                results.append({
                    'p': p, 'd': d, 'q': q,
                    'train_MAE': np.mean(train_mae_list),
                    'test_MAE': np.mean(test_mae_list)
                })

# -----------------------------
# 7️⃣ 결과 정리 및 최적 모델 출력
# -----------------------------
results_df = pd.DataFrame(results)
best = results_df.loc[results_df['test_MAE'].idxmin()]

print("✅ 최적 ARIMAX(p,d,q):", tuple(best[['p', 'd', 'q']]))
print("📉 평균 Train MAE:", round(best['train_MAE'], 4))
print("📈 평균 Test MAE:", round(best['test_MAE'], 4))

✅ 최적 ARIMAX(p,d,q): (3.0, 1.0, 1.0)
📉 평균 Train MAE: 1.0045
📈 평균 Test MAE: 3.5891


In [ ]:
import pandas as pd
import numpy as np
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_absolute_error
import warnings
warnings.filterwarnings("ignore")

df = pd.read_csv('기계장비lag.csv')

# 분기형 인덱스로 설정
df['index'] = pd.PeriodIndex(df['index'], freq='Q')
df = df.set_index('index')

# -----------------------------
# 2️⃣ 예측 대상(y)과 외생 변수(X) 설정
# -----------------------------
target = 'gdp'   # GDP 예측 대상 칼럼
exog_vars = ['생산지수','내수출하지수','매출실적BSI','업황실적BSI','수출실적BSI','생산실적BSI','신규수주실적BSI',
             '가동률실적BSI','인력사정실적BSI'
]

y = df[target].dropna()
X = df[exog_vars].dropna()

# y와 X의 인덱스 동기화
df_model = pd.concat([y, X], axis=1).dropna()
y = df_model[target]
X = df_model[exog_vars]

# -----------------------------
# 3️⃣ 롤링 예측 설정
# -----------------------------
periods_per_year = 4
train_years = 5
train_window = train_years * periods_per_year

# 예측 구간: 2017Q1 이후부터 2025Q2까지
start_period = pd.Period('2016Q1', freq='Q')
end_period = pd.Period('2025Q2', freq='Q')

# -----------------------------
# 4️⃣ 파라미터 범위 설정
# -----------------------------
p_values = range(0, 4)
d_values = range(1, 4)
q_values = range(0, 4)

# -----------------------------
# 5️⃣ 결과 저장
# -----------------------------
results = []

# -----------------------------
# 6️⃣ (p,d,q) 조합별 탐색
# -----------------------------
for p in p_values:
    for d in d_values:
        for q in q_values:

            train_mae_list = []
            test_mae_list = []

            for test_start in pd.period_range(start_period, end_period, freq='Q'):
                train_end = test_start - 1
                test_end = test_start
                train_start = train_end - (train_window - 1)

                train_data = y.loc[train_start:train_end]
                test_data = y.loc[test_end:test_end]
                X_train = X.loc[train_start:train_end]
                X_test = X.loc[test_end:test_end]

                if len(train_data) < train_window or len(X_test) == 0:
                    continue

                try:
                    model = SARIMAX(train_data, exog=X_train, order=(p, d, q),
                                    enforce_stationarity=False, enforce_invertibility=False)
                    fit = model.fit(disp=False)

                    # 예측
                    pred = fit.forecast(steps=1, exog=X_test)

                    # Train MAE (fittedvalues vs train_data)
                    train_mae = mean_absolute_error(train_data, fit.fittedvalues)
                    train_mae_list.append(train_mae)

                    # Test MAE (out-of-sample)
                    test_mae = mean_absolute_error(test_data, pred)
                    test_mae_list.append(test_mae)

                except Exception:
                    continue

            if train_mae_list and test_mae_list:
                results.append({
                    'p': p, 'd': d, 'q': q,
                    'train_MAE': np.mean(train_mae_list),
                    'test_MAE': np.mean(test_mae_list)
                })

# -----------------------------
# 7️⃣ 결과 정리 및 최적 모델 출력
# -----------------------------
results_df = pd.DataFrame(results)
best = results_df.loc[results_df['test_MAE'].idxmin()]

print("✅ 최적 ARIMAX(p,d,q):", tuple(best[['p', 'd', 'q']]))
print("📉 평균 Train MAE:", round(best['train_MAE'], 4))
print("📈 평균 Test MAE:", round(best['test_MAE'], 4))

✅ 최적 ARIMAX(p,d,q): (2.0, 1.0, 2.0)
📉 평균 Train MAE: 1.1123
📈 평균 Test MAE: 2.678


In [ ]:
import pandas as pd
import numpy as np
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_absolute_error
import warnings
warnings.filterwarnings("ignore")

df = pd.read_csv('기계장비lag.csv')

# 분기형 인덱스로 설정
df['index'] = pd.PeriodIndex(df['index'], freq='Q')
df = df.set_index('index')

# -----------------------------
# 2️⃣ 예측 대상(y)과 외생 변수(X) 설정
# -----------------------------
target = 'gdp'   # GDP 예측 대상 칼럼
exog_vars = ['생산지수','내수출하지수','매출실적BSI','업황실적BSI','수출실적BSI','생산실적BSI','신규수주실적BSI','인력사정실적BSI'
]

y = df[target].dropna()
X = df[exog_vars].dropna()

# y와 X의 인덱스 동기화
df_model = pd.concat([y, X], axis=1).dropna()
y = df_model[target]
X = df_model[exog_vars]

# -----------------------------
# 3️⃣ 롤링 예측 설정
# -----------------------------
periods_per_year = 4
train_years = 5
train_window = train_years * periods_per_year

# 예측 구간: 2017Q1 이후부터 2025Q2까지
start_period = pd.Period('2016Q1', freq='Q')
end_period = pd.Period('2025Q2', freq='Q')

# -----------------------------
# 4️⃣ 파라미터 범위 설정
# -----------------------------
p_values = range(0, 4)
d_values = range(1, 4)
q_values = range(0, 4)

# -----------------------------
# 5️⃣ 결과 저장
# -----------------------------
results = []

# -----------------------------
# 6️⃣ (p,d,q) 조합별 탐색
# -----------------------------
for p in p_values:
    for d in d_values:
        for q in q_values:

            train_mae_list = []
            test_mae_list = []

            for test_start in pd.period_range(start_period, end_period, freq='Q'):
                train_end = test_start - 1
                test_end = test_start
                train_start = train_end - (train_window - 1)

                train_data = y.loc[train_start:train_end]
                test_data = y.loc[test_end:test_end]
                X_train = X.loc[train_start:train_end]
                X_test = X.loc[test_end:test_end]

                if len(train_data) < train_window or len(X_test) == 0:
                    continue

                try:
                    model = SARIMAX(train_data, exog=X_train, order=(p, d, q),
                                    enforce_stationarity=False, enforce_invertibility=False)
                    fit = model.fit(disp=False)

                    # 예측
                    pred = fit.forecast(steps=1, exog=X_test)

                    # Train MAE (fittedvalues vs train_data)
                    train_mae = mean_absolute_error(train_data, fit.fittedvalues)
                    train_mae_list.append(train_mae)

                    # Test MAE (out-of-sample)
                    test_mae = mean_absolute_error(test_data, pred)
                    test_mae_list.append(test_mae)

                except Exception:
                    continue

            if train_mae_list and test_mae_list:
                results.append({
                    'p': p, 'd': d, 'q': q,
                    'train_MAE': np.mean(train_mae_list),
                    'test_MAE': np.mean(test_mae_list)
                })

# -----------------------------
# 7️⃣ 결과 정리 및 최적 모델 출력
# -----------------------------
results_df = pd.DataFrame(results)
best = results_df.loc[results_df['test_MAE'].idxmin()]

print("✅ 최적 ARIMAX(p,d,q):", tuple(best[['p', 'd', 'q']]))
print("📉 평균 Train MAE:", round(best['train_MAE'], 4))
print("📈 평균 Test MAE:", round(best['test_MAE'], 4))

✅ 최적 ARIMAX(p,d,q): (2.0, 1.0, 2.0)
📉 평균 Train MAE: 1.2208
📈 평균 Test MAE: 2.2326


In [ ]:
import pandas as pd
import numpy as np
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_absolute_error
import warnings
warnings.filterwarnings("ignore")

df = pd.read_csv('기계장비lag.csv')

# 분기형 인덱스로 설정
df['index'] = pd.PeriodIndex(df['index'], freq='Q')
df = df.set_index('index')

# -----------------------------
# 2️⃣ 예측 대상(y)과 외생 변수(X) 설정
# -----------------------------
target = 'gdp'   # GDP 예측 대상 칼럼
exog_vars = ['생산지수','내수출하지수','매출실적BSI','업황실적BSI','수출실적BSI','생산실적BSI','인력사정실적BSI'
]

y = df[target].dropna()
X = df[exog_vars].dropna()

# y와 X의 인덱스 동기화
df_model = pd.concat([y, X], axis=1).dropna()
y = df_model[target]
X = df_model[exog_vars]

# -----------------------------
# 3️⃣ 롤링 예측 설정
# -----------------------------
periods_per_year = 4
train_years = 5
train_window = train_years * periods_per_year

# 예측 구간: 2017Q1 이후부터 2025Q2까지
start_period = pd.Period('2016Q1', freq='Q')
end_period = pd.Period('2025Q2', freq='Q')

# -----------------------------
# 4️⃣ 파라미터 범위 설정
# -----------------------------
p_values = range(0, 4)
d_values = range(1, 4)
q_values = range(0, 4)

# -----------------------------
# 5️⃣ 결과 저장
# -----------------------------
results = []

# -----------------------------
# 6️⃣ (p,d,q) 조합별 탐색
# -----------------------------
for p in p_values:
    for d in d_values:
        for q in q_values:

            train_mae_list = []
            test_mae_list = []

            for test_start in pd.period_range(start_period, end_period, freq='Q'):
                train_end = test_start - 1
                test_end = test_start
                train_start = train_end - (train_window - 1)

                train_data = y.loc[train_start:train_end]
                test_data = y.loc[test_end:test_end]
                X_train = X.loc[train_start:train_end]
                X_test = X.loc[test_end:test_end]

                if len(train_data) < train_window or len(X_test) == 0:
                    continue

                try:
                    model = SARIMAX(train_data, exog=X_train, order=(p, d, q),
                                    enforce_stationarity=False, enforce_invertibility=False)
                    fit = model.fit(disp=False)

                    # 예측
                    pred = fit.forecast(steps=1, exog=X_test)

                    # Train MAE (fittedvalues vs train_data)
                    train_mae = mean_absolute_error(train_data, fit.fittedvalues)
                    train_mae_list.append(train_mae)

                    # Test MAE (out-of-sample)
                    test_mae = mean_absolute_error(test_data, pred)
                    test_mae_list.append(test_mae)

                except Exception:
                    continue

            if train_mae_list and test_mae_list:
                results.append({
                    'p': p, 'd': d, 'q': q,
                    'train_MAE': np.mean(train_mae_list),
                    'test_MAE': np.mean(test_mae_list)
                })

# -----------------------------
# 7️⃣ 결과 정리 및 최적 모델 출력
# -----------------------------
results_df = pd.DataFrame(results)
best = results_df.loc[results_df['test_MAE'].idxmin()]

print("✅ 최적 ARIMAX(p,d,q):", tuple(best[['p', 'd', 'q']]))
print("📉 평균 Train MAE:", round(best['train_MAE'], 4))
print("📈 평균 Test MAE:", round(best['test_MAE'], 4))

✅ 최적 ARIMAX(p,d,q): (1.0, 1.0, 0.0)
📉 평균 Train MAE: 1.3791
📈 평균 Test MAE: 2.488


In [ ]:
import pandas as pd
import numpy as np
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_absolute_error
import warnings
warnings.filterwarnings("ignore")

df = pd.read_csv('기계장비lag.csv')

# 분기형 인덱스로 설정
df['index'] = pd.PeriodIndex(df['index'], freq='Q')
df = df.set_index('index')

# -----------------------------
# 2️⃣ 예측 대상(y)과 외생 변수(X) 설정
# -----------------------------
target = 'gdp'   # GDP 예측 대상 칼럼
exog_vars = ['생산지수','내수출하지수','매출실적BSI','업황실적BSI','수출실적BSI','인력사정실적BSI'
]

y = df[target].dropna()
X = df[exog_vars].dropna()

# y와 X의 인덱스 동기화
df_model = pd.concat([y, X], axis=1).dropna()
y = df_model[target]
X = df_model[exog_vars]

# -----------------------------
# 3️⃣ 롤링 예측 설정
# -----------------------------
periods_per_year = 4
train_years = 5
train_window = train_years * periods_per_year

# 예측 구간: 2017Q1 이후부터 2025Q2까지
start_period = pd.Period('2016Q1', freq='Q')
end_period = pd.Period('2025Q2', freq='Q')

# -----------------------------
# 4️⃣ 파라미터 범위 설정
# -----------------------------
p_values = range(0, 4)
d_values = range(1, 4)
q_values = range(0, 4)

# -----------------------------
# 5️⃣ 결과 저장
# -----------------------------
results = []

# -----------------------------
# 6️⃣ (p,d,q) 조합별 탐색
# -----------------------------
for p in p_values:
    for d in d_values:
        for q in q_values:

            train_mae_list = []
            test_mae_list = []

            for test_start in pd.period_range(start_period, end_period, freq='Q'):
                train_end = test_start - 1
                test_end = test_start
                train_start = train_end - (train_window - 1)

                train_data = y.loc[train_start:train_end]
                test_data = y.loc[test_end:test_end]
                X_train = X.loc[train_start:train_end]
                X_test = X.loc[test_end:test_end]

                if len(train_data) < train_window or len(X_test) == 0:
                    continue

                try:
                    model = SARIMAX(train_data, exog=X_train, order=(p, d, q),
                                    enforce_stationarity=False, enforce_invertibility=False)
                    fit = model.fit(disp=False)

                    # 예측
                    pred = fit.forecast(steps=1, exog=X_test)

                    # Train MAE (fittedvalues vs train_data)
                    train_mae = mean_absolute_error(train_data, fit.fittedvalues)
                    train_mae_list.append(train_mae)

                    # Test MAE (out-of-sample)
                    test_mae = mean_absolute_error(test_data, pred)
                    test_mae_list.append(test_mae)

                except Exception:
                    continue

            if train_mae_list and test_mae_list:
                results.append({
                    'p': p, 'd': d, 'q': q,
                    'train_MAE': np.mean(train_mae_list),
                    'test_MAE': np.mean(test_mae_list)
                })

# -----------------------------
# 7️⃣ 결과 정리 및 최적 모델 출력
# -----------------------------
results_df = pd.DataFrame(results)
best = results_df.loc[results_df['test_MAE'].idxmin()]

print("✅ 최적 ARIMAX(p,d,q):", tuple(best[['p', 'd', 'q']]))
print("📉 평균 Train MAE:", round(best['train_MAE'], 4))
print("📈 평균 Test MAE:", round(best['test_MAE'], 4))

✅ 최적 ARIMAX(p,d,q): (2.0, 1.0, 0.0)
📉 평균 Train MAE: 1.3138
📈 평균 Test MAE: 1.934


# 랜덤포레스트

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
import warnings
warnings.filterwarnings("ignore")

# -----------------------------
# 1️⃣ 데이터 불러오기
# -----------------------------
df = pd.read_csv('기계장비lag.csv')

# 인덱스 처리 (index 컬럼 없을 경우 자동 생성)
if 'index' not in df.columns:
    df.index = pd.period_range(start='2016Q1', periods=len(df), freq='Q')
else:
    df['index'] = pd.PeriodIndex(df['index'], freq='Q')
    df = df.set_index('index')

# -----------------------------
# 2️⃣ 예측 대상(y)과 외생 변수(X)
# -----------------------------
target = 'gdp'
exog_vars = ['생산지수','내수출하지수','매출실적BSI','업황실적BSI','수출실적BSI','인력사정실적BSI']

y = df[target].dropna()
X = df[exog_vars].dropna()

# y, X 동기화
df_model = pd.concat([y, X], axis=1).dropna()
y = df_model[target]
X = df_model[exog_vars]

# -----------------------------
# 3️⃣ 롤링 윈도우 설정
# -----------------------------
periods_per_year = 4
train_years = 5
train_window = train_years * periods_per_year

start_period = pd.Period('2016Q1', freq='Q')
end_period = pd.Period('2025Q2', freq='Q')

# -----------------------------
# 4️⃣ 결과 저장
# -----------------------------
train_mae_list = []
test_mae_list = []

# -----------------------------
# 5️⃣ 롤링 예측 실행
# -----------------------------
for test_start in pd.period_range(start_period, end_period, freq='Q'):
    train_end = test_start - 1
    test_end = test_start
    train_start = train_end - (train_window - 1)

    y_train = y.loc[train_start:train_end]
    y_test = y.loc[test_end:test_end]
    X_train = X.loc[train_start:train_end]
    X_test = X.loc[test_end:test_end]

    if len(y_train) < train_window or len(X_test) == 0:
        continue

    try:
        # 랜덤포레스트 모델 학습
        rf = RandomForestRegressor(
            n_estimators=500,     # 트리 개수
            max_depth=None,      # 깊이 제한 없음
            random_state=42
        )
        rf.fit(X_train, y_train)

        # 학습 구간 예측 (Train MAE)
        train_pred = rf.predict(X_train)
        train_mae = mean_absolute_error(y_train, train_pred)
        train_mae_list.append(train_mae)

        # 테스트 구간 예측 (Test MAE)
        test_pred = rf.predict(X_test)
        test_mae = mean_absolute_error(y_test, test_pred)
        test_mae_list.append(test_mae)

    except Exception:
        continue

# -----------------------------
# 6️⃣ 결과 요약 출력
# -----------------------------
print("✅ 랜덤포레스트 롤링 예측 결과")
print("📉 평균 Train MAE:", round(np.mean(train_mae_list), 4))
print("📈 평균 Test MAE:", round(np.mean(test_mae_list), 4))


✅ 랜덤포레스트 롤링 예측 결과
📉 평균 Train MAE: 0.6116
📈 평균 Test MAE: 1.973


In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
import warnings
warnings.filterwarnings("ignore")

# -----------------------------
# 1️⃣ 데이터 불러오기
# -----------------------------
df = pd.read_csv('기계장비lag.csv')

# 인덱스 처리 (index 컬럼 없을 경우 자동 생성)
if 'index' not in df.columns:
    df.index = pd.period_range(start='2016Q1', periods=len(df), freq='Q')
else:
    df['index'] = pd.PeriodIndex(df['index'], freq='Q')
    df = df.set_index('index')

# -----------------------------
# 2️⃣ 예측 대상(y)과 외생 변수(X)
# -----------------------------
target = 'gdp'
exog_vars = ['생산지수','내수출하지수','매출실적BSI','업황실적BSI','수출실적BSI','생산실적BSI','인력사정실적BSI']

y = df[target].dropna()
X = df[exog_vars].dropna()

# y, X 동기화
df_model = pd.concat([y, X], axis=1).dropna()
y = df_model[target]
X = df_model[exog_vars]

# -----------------------------
# 3️⃣ 롤링 윈도우 설정
# -----------------------------
periods_per_year = 4
train_years = 5
train_window = train_years * periods_per_year

start_period = pd.Period('2016Q1', freq='Q')
end_period = pd.Period('2025Q2', freq='Q')

# -----------------------------
# 4️⃣ 결과 저장
# -----------------------------
train_mae_list = []
test_mae_list = []

# -----------------------------
# 5️⃣ 롤링 예측 실행
# -----------------------------
for test_start in pd.period_range(start_period, end_period, freq='Q'):
    train_end = test_start - 1
    test_end = test_start
    train_start = train_end - (train_window - 1)

    y_train = y.loc[train_start:train_end]
    y_test = y.loc[test_end:test_end]
    X_train = X.loc[train_start:train_end]
    X_test = X.loc[test_end:test_end]

    if len(y_train) < train_window or len(X_test) == 0:
        continue

    try:
        # 랜덤포레스트 모델 학습
        rf = RandomForestRegressor(
            n_estimators=500,     # 트리 개수
            max_depth=None,      # 깊이 제한 없음
            random_state=42
        )
        rf.fit(X_train, y_train)

        # 학습 구간 예측 (Train MAE)
        train_pred = rf.predict(X_train)
        train_mae = mean_absolute_error(y_train, train_pred)
        train_mae_list.append(train_mae)

        # 테스트 구간 예측 (Test MAE)
        test_pred = rf.predict(X_test)
        test_mae = mean_absolute_error(y_test, test_pred)
        test_mae_list.append(test_mae)

    except Exception:
        continue

# -----------------------------
# 6️⃣ 결과 요약 출력
# -----------------------------
print("✅ 랜덤포레스트 롤링 예측 결과")
print("📉 평균 Train MAE:", round(np.mean(train_mae_list), 4))
print("📈 평균 Test MAE:", round(np.mean(test_mae_list), 4))

✅ 랜덤포레스트 롤링 예측 결과
📉 평균 Train MAE: 0.6018
📈 평균 Test MAE: 1.9458


In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import mean_absolute_error
import warnings
warnings.filterwarnings("ignore")

# -----------------------------
# 1️⃣ 데이터 불러오기
# -----------------------------
df = pd.read_csv('기계장비lag.csv')

# 인덱스 설정
df['index'] = pd.PeriodIndex(df['index'], freq='Q')
df = df.set_index('index')

# -----------------------------
# 2️⃣ 예측 대상(y)과 외생 변수(X)
# -----------------------------
target = 'gdp'
exog_vars = ['생산지수','내수출하지수','매출실적BSI','업황실적BSI','수출실적BSI','인력사정실적BSI'
]

y = df[target].dropna()
X = df[exog_vars].dropna()

# y와 X 동기화
df_model = pd.concat([y, X], axis=1).dropna()
y = df_model[target]
X = df_model[exog_vars]

# -----------------------------
# 3️⃣ 롤링 윈도우 설정
# -----------------------------
periods_per_year = 4
train_years = 5
train_window = train_years * periods_per_year

start_period = pd.Period('2016Q1', freq='Q')
end_period = pd.Period('2025Q2', freq='Q')

# -----------------------------
# 4️⃣ 파라미터 후보 & RandomizedSearchCV
# -----------------------------
param_grid = {
    'n_estimators': [200, 400],
    'max_depth': [8, 10, 12],
    'min_samples_split': [2, 5],
    'max_features': ['sqrt']
}

# 첫 번째 윈도우 구간 설정
first_train_end = start_period + (train_years * periods_per_year - 1)
first_X_train = X.loc[start_period:first_train_end]
first_y_train = y.loc[start_period:first_train_end]

# 하이퍼파라미터 탐색 (단 1회)
rf = RandomForestRegressor(random_state=42)
rand_search = RandomizedSearchCV(
    rf,
    param_distributions=param_grid,
    n_iter=10,            # 108조합 중 10개만 탐색
    cv=3,
    scoring='neg_mean_absolute_error',
    n_jobs=-1,
    random_state=42
)
rand_search.fit(first_X_train, first_y_train)
best_params = rand_search.best_params_
print("✅ 최적 하이퍼파라미터:", best_params)

# -----------------------------
# 5️⃣ 롤링 예측 (최적 파라미터 고정)
# -----------------------------
train_mae_list = []
test_mae_list = []

for test_start in pd.period_range(start_period, end_period, freq='Q'):
    train_end = test_start - 1
    test_end = test_start
    train_start = train_end - (train_window - 1)

    y_train = y.loc[train_start:train_end]
    y_test = y.loc[test_end:test_end]
    X_train = X.loc[train_start:train_end]
    X_test = X.loc[test_end:test_end]

    if len(y_train) < train_window or len(X_test) == 0:
        continue

    try:
        # 고정된 최적 파라미터로 학습
        model = RandomForestRegressor(**best_params, random_state=42)
        model.fit(X_train, y_train)

        # 예측
        train_pred = model.predict(X_train)
        test_pred = model.predict(X_test)

        # MAE 계산
        train_mae = mean_absolute_error(y_train, train_pred)
        test_mae = mean_absolute_error(y_test, test_pred)

        train_mae_list.append(train_mae)
        test_mae_list.append(test_mae)

    except Exception:
        continue

# -----------------------------
# 6️⃣ 결과 요약
# -----------------------------
print("\n📊 랜덤포레스트 롤링 예측 결과")
print(f"📉 평균 Train MAE: {np.mean(train_mae_list):.4f}")
print(f"📈 평균 Test MAE: {np.mean(test_mae_list):.4f}")

✅ 최적 하이퍼파라미터: {'n_estimators': 400, 'min_samples_split': 5, 'max_features': 'sqrt', 'max_depth': 12}

📊 랜덤포레스트 롤링 예측 결과
📉 평균 Train MAE: 0.7902
📈 평균 Test MAE: 1.9087


In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import mean_absolute_error
import warnings
warnings.filterwarnings("ignore")

# -----------------------------
# 1️⃣ 데이터 불러오기
# -----------------------------
df = pd.read_csv('기계장비lag.csv')

# 인덱스 설정
df['index'] = pd.PeriodIndex(df['index'], freq='Q')
df = df.set_index('index')

# -----------------------------
# 2️⃣ 예측 대상(y)과 외생 변수(X)
# -----------------------------
target = 'gdp'
exog_vars = ['생산지수','내수출하지수','매출실적BSI','업황실적BSI','수출실적BSI'
]

y = df[target].dropna()
X = df[exog_vars].dropna()

# y와 X 동기화
df_model = pd.concat([y, X], axis=1).dropna()
y = df_model[target]
X = df_model[exog_vars]

# -----------------------------
# 3️⃣ 롤링 윈도우 설정
# -----------------------------
periods_per_year = 4
train_years = 5
train_window = train_years * periods_per_year

start_period = pd.Period('2016Q1', freq='Q')
end_period = pd.Period('2025Q2', freq='Q')

# -----------------------------
# 4️⃣ 파라미터 후보 & RandomizedSearchCV
# -----------------------------
param_grid = {
    'n_estimators': [200, 400],
    'max_depth': [8, 10, 12],
    'min_samples_split': [2, 5],
    'max_features': ['sqrt']
}

# 첫 번째 윈도우 구간 설정
first_train_end = start_period + (train_years * periods_per_year - 1)
first_X_train = X.loc[start_period:first_train_end]
first_y_train = y.loc[start_period:first_train_end]

# 하이퍼파라미터 탐색 (단 1회)
rf = RandomForestRegressor(random_state=42)
rand_search = RandomizedSearchCV(
    rf,
    param_distributions=param_grid,
    n_iter=10,            # 108조합 중 10개만 탐색
    cv=3,
    scoring='neg_mean_absolute_error',
    n_jobs=-1,
    random_state=42
)
rand_search.fit(first_X_train, first_y_train)
best_params = rand_search.best_params_
print("✅ 최적 하이퍼파라미터:", best_params)

# -----------------------------
# 5️⃣ 롤링 예측 (최적 파라미터 고정)
# -----------------------------
train_mae_list = []
test_mae_list = []

for test_start in pd.period_range(start_period, end_period, freq='Q'):
    train_end = test_start - 1
    test_end = test_start
    train_start = train_end - (train_window - 1)

    y_train = y.loc[train_start:train_end]
    y_test = y.loc[test_end:test_end]
    X_train = X.loc[train_start:train_end]
    X_test = X.loc[test_end:test_end]

    if len(y_train) < train_window or len(X_test) == 0:
        continue

    try:
        # 고정된 최적 파라미터로 학습
        model = RandomForestRegressor(**best_params, random_state=42)
        model.fit(X_train, y_train)

        # 예측
        train_pred = model.predict(X_train)
        test_pred = model.predict(X_test)

        # MAE 계산
        train_mae = mean_absolute_error(y_train, train_pred)
        test_mae = mean_absolute_error(y_test, test_pred)

        train_mae_list.append(train_mae)
        test_mae_list.append(test_mae)

    except Exception:
        continue

# -----------------------------
# 6️⃣ 결과 요약
# -----------------------------
print("\n📊 랜덤포레스트 롤링 예측 결과")
print(f"📉 평균 Train MAE: {np.mean(train_mae_list):.4f}")
print(f"📈 평균 Test MAE: {np.mean(test_mae_list):.4f}")

✅ 최적 하이퍼파라미터: {'n_estimators': 400, 'min_samples_split': 5, 'max_features': 'sqrt', 'max_depth': 12}

📊 랜덤포레스트 롤링 예측 결과
📉 평균 Train MAE: 0.8196
📈 평균 Test MAE: 2.0145


#AR

In [ ]:
import pandas as pd
import numpy as np
from statsmodels.tsa.ar_model import AutoReg
from sklearn.metrics import mean_absolute_error
import warnings
warnings.filterwarnings("ignore")

# -----------------------------
# 1️⃣ 데이터 불러오기
# -----------------------------
df = pd.read_csv('기계장비qoq.csv')₩

# GDP 단일 시계열만 사용 (AR(1))
y = df['gdp'].dropna()
y.index = pd.PeriodIndex(df.loc[y.index, 'index'], freq='Q')

# -----------------------------
# 2️⃣ 롤링 윈도우 설정
# -----------------------------
periods_per_year = 4
train_years = 5
train_window = train_years * periods_per_year

start_period = pd.Period('2016Q1', freq='Q')
end_period = pd.Period('2025Q2', freq='Q')

train_mae_list = []
test_mae_list = []

# -----------------------------
# 3️⃣ 롤링 예측 실행
# -----------------------------
for test_start in pd.period_range(start_period, end_period, freq='Q'):
    train_end = test_start - 1
    test_end = test_start
    train_start = train_end - (train_window - 1)

    train_data = y.loc[train_start:train_end]
    test_data = y.loc[test_end:test_end]

    if len(train_data) < train_window:
        continue

    try:
        # AR(1) 학습
        model = AutoReg(train_data, lags=1, old_names=False)
        fit = model.fit()

        # --- Train 예측 ---
        # fittedvalues는 길이가 n-1이므로 첫 번째 값 제외하고 MAE 계산
        train_mae = mean_absolute_error(train_data[1:], fit.fittedvalues)
        train_mae_list.append(train_mae)

        # --- Test 예측 (1분기 앞 예측) ---
        pred = fit.predict(start=len(train_data), end=len(train_data))
        test_mae = mean_absolute_error(test_data, pred)
        test_mae_list.append(test_mae)

    except Exception as e:
        print("❌ Error:", e)
        continue

# -----------------------------
# 4️⃣ 결과 출력
# -----------------------------
print("✅ AR(1) 롤링 예측 결과")
print(f"📉 평균 Train MAE: {np.mean(train_mae_list):.4f}")
print(f"📈 평균 Test MAE: {np.mean(test_mae_list):.4f}")


✅ AR(1) 롤링 예측 결과
📉 평균 Train MAE: 2.9329
📈 평균 Test MAE: 2.8062


#XGB

In [ ]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import mean_absolute_error
import warnings
warnings.filterwarnings("ignore")

# -----------------------------
# 1️⃣ 데이터 불러오기
# -----------------------------
df = pd.read_csv('기계장비lag.csv')

# 인덱스 설정
df['index'] = pd.PeriodIndex(df['index'], freq='Q')
df = df.set_index('index')

# -----------------------------
# 2️⃣ 예측 대상(y)과 외생 변수(X)
# -----------------------------
target = 'gdp'
exog_vars = ['생산지수','내수출하지수','매출실적BSI','업황실적BSI','수출실적BSI','인력사정실적BSI'
]

y = df[target].dropna()
X = df[exog_vars].dropna()

# y와 X 동기화
df_model = pd.concat([y, X], axis=1).dropna()
y = df_model[target]
X = df_model[exog_vars]

# -----------------------------
# 3️⃣ 롤링 윈도우 설정
# -----------------------------
periods_per_year = 4
train_years = 5
train_window = train_years * periods_per_year

start_period = pd.Period('2016Q1', freq='Q')
end_period = pd.Period('2025Q2', freq='Q')

# -----------------------------
# 4️⃣ 하이퍼파라미터 후보 & RandomizedSearchCV
# -----------------------------
param_grid = {
    'n_estimators': [200, 400, 600],
    'learning_rate': [0.03, 0.05, 0.1],
    'max_depth': [3, 5, 7, 9],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0],
    'min_child_weight': [1, 3, 5],
    'gamma': [0, 0.1, 0.3]
}

# 첫 번째 윈도우로 탐색
first_train_end = start_period + (train_years * periods_per_year - 1)
first_X_train = X.loc[start_period:first_train_end]
first_y_train = y.loc[start_period:first_train_end]

# RandomizedSearchCV 실행
xgb = XGBRegressor(random_state=42, objective='reg:squarederror')
rand_search = RandomizedSearchCV(
    xgb,
    param_distributions=param_grid,
    n_iter=15,          # 전체 중 15조합 탐색
    cv=3,
    scoring='neg_mean_absolute_error',
    n_jobs=-1,
    random_state=42
)
rand_search.fit(first_X_train, first_y_train)
best_params = rand_search.best_params_
print("✅ 최적 하이퍼파라미터:", best_params)

# -----------------------------
# 5️⃣ 롤링 예측 (최적 파라미터 고정)
# -----------------------------
train_mae_list = []
test_mae_list = []

for test_start in pd.period_range(start_period, end_period, freq='Q'):
    train_end = test_start - 1
    test_end = test_start
    train_start = train_end - (train_window - 1)

    y_train = y.loc[train_start:train_end]
    y_test = y.loc[test_end:test_end]
    X_train = X.loc[train_start:train_end]
    X_test = X.loc[test_end:test_end]

    if len(y_train) < train_window or len(X_test) == 0:
        continue

    try:
        # 고정된 최적 파라미터로 학습
        model = XGBRegressor(**best_params, random_state=42, objective='reg:squarederror')
        model.fit(X_train, y_train)

        # 예측
        train_pred = model.predict(X_train)
        test_pred = model.predict(X_test)

        # MAE 계산
        train_mae = mean_absolute_error(y_train, train_pred)
        test_mae = mean_absolute_error(y_test, test_pred)

        train_mae_list.append(train_mae)
        test_mae_list.append(test_mae)

    except Exception as e:
        print("❌ Error:", e)
        continue

# -----------------------------
# 6️⃣ 결과 요약
# -----------------------------
print("\n📊 XGBoost 롤링 예측 결과")
print(f"📉 평균 Train MAE: {np.mean(train_mae_list):.4f}")
print(f"📈 평균 Test MAE: {np.mean(test_mae_list):.4f}")


✅ 최적 하이퍼파라미터: {'subsample': 0.8, 'n_estimators': 600, 'min_child_weight': 3, 'max_depth': 5, 'learning_rate': 0.05, 'gamma': 0.1, 'colsample_bytree': 1.0}

📊 XGBoost 롤링 예측 결과
📉 평균 Train MAE: 0.1513
📈 평균 Test MAE: 2.0730


#MLP

In [ ]:
import pandas as pd
import numpy as np
from sklearn.neural_network import MLPRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings("ignore")

# -----------------------------
# 1️⃣ 데이터 불러오기
# -----------------------------
df = pd.read_csv('기계장비lag.csv')

# 인덱스 설정
df['index'] = pd.PeriodIndex(df['index'], freq='Q')
df = df.set_index('index')

# -----------------------------
# 2️⃣ 예측 대상(y)과 외생 변수(X)
# -----------------------------
target = 'gdp'
exog_vars = ['생산지수','내수출하지수','매출실적BSI','업황실적BSI','수출실적BSI','인력사정실적BSI'
]

y = df[target].dropna()
X = df[exog_vars].dropna()

# y와 X 동기화
df_model = pd.concat([y, X], axis=1).dropna()
y = df_model[target]
X = df_model[exog_vars]

# 스케일링 (NN에서는 매우 중요)
scaler_X = StandardScaler()
scaler_y = StandardScaler()
X_scaled = pd.DataFrame(scaler_X.fit_transform(X), index=X.index, columns=X.columns)
y_scaled = pd.Series(scaler_y.fit_transform(y.values.reshape(-1, 1)).flatten(), index=y.index)

# -----------------------------
# 3️⃣ 롤링 윈도우 설정
# -----------------------------
periods_per_year = 4
train_years = 5
train_window = train_years * periods_per_year

start_period = pd.Period('2016Q1', freq='Q')
end_period = pd.Period('2025Q2', freq='Q')

# -----------------------------
# 4️⃣ 하이퍼파라미터 후보 & RandomizedSearchCV
# -----------------------------
param_grid = {
    'hidden_layer_sizes': [(32,), (64,), (32,16), (64,32), (128,64)],
    'activation': ['relu', 'tanh'],
    'solver': ['adam', 'lbfgs'],
    'alpha': [0.0001, 0.001, 0.01],
    'learning_rate_init': [0.001, 0.01]
}

# 첫 번째 윈도우로 하이퍼파라미터 탐색
first_train_end = start_period + (train_years * periods_per_year - 1)
first_X_train = X_scaled.loc[start_period:first_train_end]
first_y_train = y_scaled.loc[start_period:first_train_end]

mlp = MLPRegressor(random_state=42, max_iter=1000)
rand_search = RandomizedSearchCV(
    mlp,
    param_distributions=param_grid,
    n_iter=10,
    cv=3,
    scoring='neg_mean_absolute_error',
    n_jobs=-1,
    random_state=42
)
rand_search.fit(first_X_train, first_y_train)
best_params = rand_search.best_params_
print("✅ 최적 하이퍼파라미터:", best_params)

# -----------------------------
# 5️⃣ 롤링 예측 (최적 파라미터 고정)
# -----------------------------
train_mae_list = []
test_mae_list = []

for test_start in pd.period_range(start_period, end_period, freq='Q'):
    train_end = test_start - 1
    test_end = test_start
    train_start = train_end - (train_window - 1)

    y_train = y_scaled.loc[train_start:train_end]
    y_test = y_scaled.loc[test_end:test_end]
    X_train = X_scaled.loc[train_start:train_end]
    X_test = X_scaled.loc[test_end:test_end]

    if len(y_train) < train_window or len(X_test) == 0:
        continue

    try:
        model = MLPRegressor(**best_params, random_state=42, max_iter=1000)
        model.fit(X_train, y_train)

        train_pred = model.predict(X_train)
        test_pred = model.predict(X_test)

        # 스케일링 복원
        train_pred_inv = scaler_y.inverse_transform(train_pred.reshape(-1, 1)).flatten()
        test_pred_inv = scaler_y.inverse_transform(test_pred.reshape(-1, 1)).flatten()
        y_train_inv = scaler_y.inverse_transform(y_train.values.reshape(-1, 1)).flatten()
        y_test_inv = scaler_y.inverse_transform(y_test.values.reshape(-1, 1)).flatten()

        # MAE 계산
        train_mae = mean_absolute_error(y_train_inv, train_pred_inv)
        test_mae = mean_absolute_error(y_test_inv, test_pred_inv)

        train_mae_list.append(train_mae)
        test_mae_list.append(test_mae)

    except Exception as e:
        print("❌ Error:", e)
        continue

# -----------------------------
# 6️⃣ 결과 요약
# -----------------------------
print("\n📊 Neural Network (MLP) 롤링 예측 결과")
print(f"📉 평균 Train MAE: {np.mean(train_mae_list):.4f}")
print(f"📈 평균 Test MAE: {np.mean(test_mae_list):.4f}")


✅ 최적 하이퍼파라미터: {'solver': 'adam', 'learning_rate_init': 0.001, 'hidden_layer_sizes': (64,), 'alpha': 0.01, 'activation': 'relu'}

📊 Neural Network (MLP) 롤링 예측 결과
📉 평균 Train MAE: 0.4995
📈 평균 Test MAE: 2.0553


#LSTM

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
import warnings
warnings.filterwarnings("ignore")

# -----------------------------
# 1️⃣ 데이터 불러오기
# -----------------------------
df = pd.read_csv('기계장비lag.csv')

# 인덱스 설정
df['index'] = pd.PeriodIndex(df['index'], freq='Q')
df = df.set_index('index')

# -----------------------------
# 2️⃣ 예측 대상(y)과 외생 변수(X)
# -----------------------------
target = 'gdp'
exog_vars = [
'생산지수','내수출하지수','매출실적BSI','업황실적BSI','수출실적BSI','인력사정실적BSI'
]

y = df[target].dropna()
X = df[exog_vars].dropna()

# y와 X 동기화
df_model = pd.concat([y, X], axis=1).dropna()
y = df_model[target]
X = df_model[exog_vars]

# -----------------------------
# 3️⃣ 스케일링
# -----------------------------
scaler_X = StandardScaler()
scaler_y = StandardScaler()
X_scaled = pd.DataFrame(scaler_X.fit_transform(X), index=X.index, columns=X.columns)
y_scaled = pd.Series(scaler_y.fit_transform(y.values.reshape(-1, 1)).flatten(), index=y.index)

# -----------------------------
# 4️⃣ LSTM 입력 형태 변환 함수
# -----------------------------
def create_lstm_input(X, y, lookback=4):
    """
    lookback: 과거 몇 분기 데이터를 사용할지 (ex. 4 = 1년)
    """
    Xs, ys, idxs = [], [], []
    for i in range(lookback, len(X)):
        Xs.append(X.iloc[i-lookback:i].values)
        ys.append(y.iloc[i])
        idxs.append(y.index[i])
    return np.array(Xs), np.array(ys), idxs

lookback = 4
X_seq, y_seq, idxs = create_lstm_input(X_scaled, y_scaled, lookback)
y_seq = pd.Series(y_seq, index=idxs)

# -----------------------------
# 5️⃣ 롤링 윈도우 설정
# -----------------------------
periods_per_year = 4
train_years = 5
train_window = train_years * periods_per_year

start_period = pd.Period('2016Q1', freq='Q')
end_period = pd.Period('2025Q2', freq='Q')

# -----------------------------
# 6️⃣ LSTM 모델 정의 함수
# -----------------------------
def build_lstm(input_shape):
    model = Sequential()
    model.add(LSTM(64, activation='tanh', input_shape=input_shape, return_sequences=False))
    model.add(Dropout(0.2))
    model.add(Dense(32, activation='relu'))
    model.add(Dense(1))
    model.compile(optimizer='adam', loss='mae')
    return model

# -----------------------------
# 7️⃣ 롤링 예측
# -----------------------------
train_mae_list, test_mae_list = [], []

for test_start in pd.period_range(start_period, end_period, freq='Q'):
    train_end = test_start - 1
    test_end = test_start
    train_start = train_end - (train_window - 1)

    # 윈도우 범위 맞추기
    train_mask = (np.array(idxs) >= train_start) & (np.array(idxs) <= train_end)
    test_mask = (np.array(idxs) == test_end)

    X_train, y_train = X_seq[train_mask], y_seq[train_mask]
    X_test, y_test = X_seq[test_mask], y_seq[test_mask]

    if len(y_train) < train_window or len(X_test) == 0:
        continue

    try:
        model = build_lstm((X_train.shape[1], X_train.shape[2]))
        early_stop = EarlyStopping(monitor='loss', patience=5, restore_best_weights=True)
        model.fit(X_train, y_train, epochs=200, batch_size=8, verbose=0, callbacks=[early_stop])

        # 예측
        train_pred = model.predict(X_train, verbose=0)
        test_pred = model.predict(X_test, verbose=0)

        # 스케일링 복원
        train_pred_inv = scaler_y.inverse_transform(train_pred)
        test_pred_inv = scaler_y.inverse_transform(test_pred)
        y_train_inv = scaler_y.inverse_transform(y_train.values.reshape(-1, 1))
        y_test_inv = scaler_y.inverse_transform(y_test.values.reshape(-1, 1))

        # MAE 계산
        train_mae = mean_absolute_error(y_train_inv, train_pred_inv)
        test_mae = mean_absolute_error(y_test_inv, test_pred_inv)

        train_mae_list.append(train_mae)
        test_mae_list.append(test_mae)

    except Exception as e:
        print("❌ Error:", e)
        continue

# -----------------------------
# 8️⃣ 결과 요약
# -----------------------------
print("\n📊 LSTM 롤링 예측 결과")
print(f"📉 평균 Train MAE: {np.mean(train_mae_list):.4f}")
print(f"📈 평균 Test MAE: {np.mean(test_mae_list):.4f}")



📊 LSTM 롤링 예측 결과
📉 평균 Train MAE: 1.1355
📈 평균 Test MAE: 2.7587
